# Notebook 05 - Thruster Model

## Purpose

Notebook 05 introduces the first explicit actuator model into the validated HydroLab-3D translational dynamics stack.

The previous notebooks established how vehicle position evolves from velocity, how vectors transform between body and world frames, how forces generate acceleration, and how hydrodynamic drag, gravity, and buoyancy affect translational motion.

Until now, externally generated motion has been represented using an abstract applied force. This notebook replaces that abstraction with a physically bounded thruster command model:

```text
normalized command
        ↓
command saturation
        ↓
thrust magnitude
        ↓
body-frame thrust direction
        ↓
body-to-world transformation
        ↓
vehicle force balance
        ↓
acceleration
        ↓
velocity
        ↓
position
```

The objective is not to construct a complete propulsion system. It is to establish the smallest trustworthy actuator layer required for later control, current-disturbance, and full vehicle-dynamics work.

---

## Why This Matters

Controllers and autonomy algorithms should not directly manipulate vehicle position, velocity, or arbitrary force.

A physically meaningful simulation requires an actuator boundary between a command and the force actually applied to the vehicle.

Without this layer, a command could implicitly request unlimited force or bypass the physical direction of a thruster. With a reduced thruster model, HydroLab-3D can represent an important causal chain:

```text
command
   ↓
actuator
   ↓
force
   ↓
vehicle dynamics
   ↓
motion
```

This creates a clean separation between future control logic and the already validated physical dynamics.

It also establishes the actuator constraints required before later notebooks introduce environmental disturbances and closed-loop control.

---

## What We Will Implement

Notebook 05 will implement and validate a reduced translational thruster model containing:

- normalized scalar thruster commands;
- command saturation;
- bounded command-to-thrust conversion;
- bidirectional thrust for positive and negative commands;
- an explicitly defined body-frame thruster direction;
- body-frame thrust-vector construction;
- transformation of thrust into the world frame using the validated orientation infrastructure;
- integration of thruster force with drag, gravity, and buoyancy;
- propagation through the existing force-to-acceleration, velocity, and position integration functions;
- analytical validation of thrust-driven terminal speed in a clean one-axis case;
- production promotion only after notebook-level validation;
- focused regression tests for the promoted actuator implementation.

The initial model will remain deliberately static.

Actuator slew limits and first-order lag are physically meaningful extensions, but they are not required to establish the fundamental command-to-force contract. They will therefore not be added unless later validation demonstrates that they are necessary for the intended reduced-order simulator behavior.

---

## What We Inherit From Notebooks 00–04

Notebook 05 builds directly on previously validated HydroLab-3D components rather than reimplementing them.

### Notebook 00 - Basic 3D Kinematics

Position propagation is already available through:

`src/hydrolab3d/dynamics/kinematics.py`

with:

`integrate_position()`

using:

$$
\mathbf{p}_{k+1}
=
\mathbf{p}_k
+
\mathbf{v}_k\Delta t
$$

### Notebook 01 - Orientation and Reference Frames

Body-to-world transformations are already available through:

`src/hydrolab3d/utils/transforms.py`

including:

`rotation_body_to_world()`

with:

$$
R_B^W
=
R_z(\psi)R_y(\theta)R_x(\phi)
$$

### Notebook 02 - Basic Dynamics

Force-driven translational dynamics are already available through:

`src/hydrolab3d/dynamics/rigid_body.py`

including:

`acceleration_from_force()`

and:

`integrate_velocity()`

with:

$$
\mathbf{a}
=
\frac{\mathbf{F}_{net}}{m}
$$

and:

$$
\mathbf{v}_{k+1}
=
\mathbf{v}_k
+
\mathbf{a}_k\Delta t
$$

### Notebook 03 - Hydrodynamic Drag

Quadratic translational drag is already available through:

`src/hydrolab3d/dynamics/drag.py`

with:

`quadratic_drag_force()`

using:

$$
\mathbf{F}_D
=
-C_D
\|\mathbf{v}\|
\mathbf{v}
$$

### Notebook 04 - Gravity and Buoyancy

Gravity and buoyancy are already available through:

`src/hydrolab3d/dynamics/buoyancy.py`

with:

`gravity_force()`

and:

`buoyancy_force()`

using:

$$
\mathbf{F}_G
=
\begin{bmatrix}
0 \\
0 \\
-mg
\end{bmatrix}
$$

and:

$$
\mathbf{F}_B
=
\begin{bmatrix}
0 \\
0 \\
\rho gV
\end{bmatrix}
$$

These components are engineering dependencies for Notebook 05. Their previously validated physics will be reused rather than reopened unless the new actuator integration exposes an actual interface or regression problem.

---

## Mathematical Model

The validated translational force balance currently has the form:

$$
m\dot{\mathbf{v}}
=
\mathbf{F}_{applied}
+
\mathbf{F}_D
+
\mathbf{F}_G
+
\mathbf{F}_B
$$

Notebook 05 replaces the abstract applied force with explicit thruster-generated force:

$$
m\dot{\mathbf{v}}
=
\mathbf{F}_T
+
\mathbf{F}_D
+
\mathbf{F}_G
+
\mathbf{F}_B
$$

The corresponding net force is therefore:

$$
\mathbf{F}_{net}
=
\mathbf{F}_T
+
\mathbf{F}_D
+
\mathbf{F}_G
+
\mathbf{F}_B
$$

and translational acceleration remains:

$$
\mathbf{a}
=
\frac{\mathbf{F}_{net}}{m}
$$

For a normalized scalar command $u$, the admissible command is:

$$
u_{sat}
=
\operatorname{clip}(u,-1,1)
$$

Given a maximum thrust magnitude $F_{max}$:

$$
F_T
=
u_{sat}F_{max}
$$

Let the thruster have a body-frame unit direction:

$$
\hat{\mathbf{d}}_B
=
\begin{bmatrix}
d_x \\
d_y \\
d_z
\end{bmatrix}
$$

with:

$$
\|\hat{\mathbf{d}}_B\|
=
1
$$

The resulting body-frame thrust vector is:

$$
\mathbf{F}_{T,B}
=
F_T\hat{\mathbf{d}}_B
$$

For a vehicle orientation described by roll $\phi$, pitch $\theta$, and yaw $\psi$, the corresponding world-frame thrust vector is:

$$
\mathbf{F}_{T,W}
=
R_B^W
\mathbf{F}_{T,B}
$$

where:

$$
R_B^W
=
R_z(\psi)R_y(\theta)R_x(\phi)
$$

The world-frame translational force balance then becomes:

$$
\mathbf{F}_{net,W}
=
\mathbf{F}_{T,W}
+
\mathbf{F}_{D,W}
+
\mathbf{F}_{G,W}
+
\mathbf{F}_{B,W}
$$

followed by:

$$
\mathbf{a}_W
=
\frac{\mathbf{F}_{net,W}}{m}
$$

---

## Thruster Command Convention

The reduced thruster command is:

$$
u\in\mathbb{R}
$$

with the physical actuator range defined by:

$$
u_{sat}\in[-1,1]
$$

The intended meanings are:

```text
u =  0      → zero commanded thrust

0 < u < 1   → proportional forward thrust

u ≥  1      → maximum forward thrust

-1 < u < 0  → proportional reverse thrust

u ≤ -1      → maximum reverse thrust
```

Thus:

$$
u=1
\Rightarrow
F_T=F_{max}
$$

and:

$$
u=-1
\Rightarrow
F_T=-F_{max}
$$

Commands outside the valid actuator range will saturate rather than produce additional force.

The thruster direction will be treated as an explicit physical parameter. It must be a finite, non-zero three-dimensional direction satisfying the documented API contract. The production implementation must not conceal physically invalid direction inputs through undocumented behavior.

---

## Reference-Frame Convention

HydroLab-3D retains its existing right-handed world-frame convention:

```text
+x_W = horizontal x
+y_W = horizontal y
+z_W = upward
```

Gravity therefore acts along negative world $z$, while buoyancy acts along positive world $z$.

Thruster orientation is specified in the vehicle body frame.

For example, a forward-facing thruster aligned with positive body $x$ has:

$$
\hat{\mathbf{d}}_B
=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

At zero vehicle orientation:

$$
R_B^W=I
$$

and therefore:

$$
\mathbf{F}_{T,W}
=
\mathbf{F}_{T,B}
$$

For a rotated vehicle, the existing body-to-world rotation matrix determines the world-frame direction of the same physical thruster.

This distinction is essential: the thruster remains attached to the vehicle, so its world-frame force direction must rotate with the vehicle.

---

## Numerical Integration Convention

Notebook 05 will retain the explicit-Euler integration sequence already established and characterized in the preceding notebooks.

For timestep $k$:

1. evaluate thruster force from the current command;
2. evaluate drag from the current velocity;
3. evaluate gravity and buoyancy;
4. sum the world-frame forces;
5. compute acceleration;
6. update velocity;
7. propagate position using the updated simulation state according to the established HydroLab-3D integration contract.

The core dynamics remain:

$$
\mathbf{a}_k
=
\frac{\mathbf{F}_{net,k}}{m}
$$

$$
\mathbf{v}_{k+1}
=
\mathbf{v}_k
+
\mathbf{a}_k\Delta t
$$

with position propagation handled by the already validated kinematic implementation.

No new integration scheme will be introduced merely for the actuator model.

---

## Expected Result

A correctly implemented thruster model should produce a transparent causal relationship between actuator command and vehicle motion.

For an axis-aligned thruster:

```text
command
   ↓
bounded thrust
   ↓
net force
   ↓
acceleration
   ↓
velocity
   ↓
position
```

At zero command, the thruster contribution should be exactly zero.

For a neutrally buoyant vehicle initially at rest with zero command, the complete force balance should produce no motion.

For a positive command applied through a forward-facing thruster, the vehicle should accelerate in the expected direction.

For a negative command, the thrust direction should reverse.

Commands whose magnitudes exceed unity should produce no more than the configured maximum thrust magnitude.

Under sustained thrust and quadratic drag, a clean one-dimensional case should approach a predictable equilibrium where:

$$
F_T+F_D=0
$$

For positive steady motion:

$$
F_T
=
C_Dv_{\infty}^2
$$

and therefore:

$$
v_{\infty}
=
\sqrt{\frac{F_T}{C_D}}
$$

This analytical equilibrium provides a strong end-to-end validation of the actuator-to-dynamics chain.

---

## Validation Criteria

Notebook 05 will be considered successful only if the reduced thruster model demonstrates the following behavior.

### Command Mapping

- zero command produces zero thrust;
- positive commands produce thrust in the configured direction;
- negative commands produce reverse thrust;
- thrust magnitude scales linearly with command inside the unsaturated region;
- commands greater than $1$ saturate at maximum forward thrust;
- commands less than $-1$ saturate at maximum reverse thrust.

### Parameter Integrity

The actuator implementation must reject physically invalid or numerically unsafe parameters, including relevant cases such as:

- non-scalar command inputs;
- non-finite commands;
- invalid maximum-thrust values;
- non-finite direction components;
- incorrectly shaped direction vectors;
- zero-magnitude thruster directions;
- violations of the documented unit-direction contract.

### Reference Frames

- an axis-aligned body-frame thruster produces the expected body-frame force;
- zero orientation preserves the thrust vector;
- known vehicle rotations produce analytically predictable world-frame thrust directions;
- rotation changes thrust direction without changing thrust magnitude.

### Integrated Dynamics

The thruster must operate correctly with the existing:

- quadratic drag;
- gravity;
- buoyancy;
- force-to-acceleration conversion;
- velocity integration;
- position propagation.

### Physical Behavior

- sustained thrust with quadratic drag approaches the expected terminal speed in a suitable one-axis experiment;
- a neutrally buoyant vehicle with zero command remains stationary;
- axis-aligned experiments produce no unintended cross-axis motion;
- saturation prevents unbounded actuator force.

### Production Integrity

After notebook-local validation:

- reusable actuator logic is promoted into the appropriate production dynamics module;
- package exports are updated only where required;
- focused automated regression tests are added;
- the production implementation is re-imported into the notebook;
- the canonical validation is repeated using production code;
- the complete repository regression suite passes.

---

## Relationship to HydroLab-3D

Notebook 05 completes the actuator side of the current reduced-order translational physics stack.

The developing architecture becomes:

```text
                command
                   ↓
              thruster model
                   ↓
             body-frame force
                   ↓
         body-to-world transform
                   ↓
                thrust
                   │
                   ├───────────────┐
                   │               │
                   ▼               ▼
                 drag       gravity + buoyancy
                   │               │
                   └───────┬───────┘
                           ↓
                       net force
                           ↓
                       acceleration
                           ↓
                        velocity
                           ↓
                        position
```

This preserves HydroLab-3D's selective-fidelity philosophy: actuator behavior is represented strongly enough to support meaningful robotics experiments without introducing unnecessary propulsion-system complexity.

Notebook 05 therefore establishes the interface through which later controllers, keyboard commands, autonomy algorithms, and learning policies can eventually influence the simulated vehicle.

---

## Scope Boundary

Notebook 05 is intentionally limited to a reduced translational actuator model.

It may include:

- normalized scalar commands;
- saturation;
- bidirectional thrust;
- thrust magnitude;
- fixed body-frame thruster direction;
- body-to-world thrust transformation;
- interaction with drag, gravity, and buoyancy;
- translational vehicle motion;
- analytical terminal-speed validation;
- production promotion;
- focused regression testing;
- scientifically useful persistent artifacts.

It will **not** introduce:

- multi-thruster allocation;
- control allocation matrices;
- actuator-generated torque;
- rotational rigid-body dynamics;
- full 6-DOF vehicle dynamics;
- added mass;
- Coriolis or centripetal terms;
- propeller blade or motor electrical dynamics;
- detailed thrust-RPM curves;
- cavitation;
- ocean currents;
- environmental disturbance models;
- closed-loop controllers;
- sensors;
- ROS 2 integration;
- reinforcement learning.

These belong to later layers of HydroLab-3D.

---

## Expected Outputs

Notebook 05 will generate only artifacts that provide useful engineering evidence.

The expected persistent outputs are:

```text
results/notebooks/05_thruster_model/
├── data/
│   └── thruster_validation.csv
└── plots/
    ├── command_thrust_response.png
    └── velocity_validation.png
```

A force-balance figure may be added only if the final integrated experiment demonstrates information that is not already clear from the numerical dataset and velocity response.

The validation dataset should contain the quantities required to reconstruct the canonical actuator-driven experiment, including relevant fields such as:

```text
time
command
thrust
drag
gravity
buoyancy
net force
acceleration
velocity
position
```

Exact artifact names and contents will be finalized by the validated implementation rather than created speculatively.

---

## Output Directory

All persistent Notebook 05 artifacts will be stored under:

`results/notebooks/05_thruster_model/`

Subdirectories will be created only when they contain actual outputs.

---

## Engineering Handoff Target

Notebook 05 should leave behind a validated actuator contract that later HydroLab-3D development can use without reopening its fundamental behavior.

After successful completion, the reduced translational model will be:

$$
m\dot{\mathbf{v}}
=
\mathbf{F}_T
+
\mathbf{F}_D
+
\mathbf{F}_G
+
\mathbf{F}_B
$$

with:

$$
\mathbf{F}_{T,W}
=
R_B^W
\left(
u_{sat}F_{max}\hat{\mathbf{d}}_B
\right)
$$

and:

$$
u_{sat}
=
\operatorname{clip}(u,-1,1)
$$

Notebook 06 may therefore assume that the following components have already been validated:

- 3D kinematic propagation;
- orientation and body-to-world transformations;
- force-to-acceleration conversion;
- velocity integration;
- quadratic hydrodynamic drag;
- gravity;
- buoyancy;
- command-to-thrust conversion;
- actuator saturation;
- thruster-direction behavior;
- transformation of body-frame thrust into world-frame force;
- integration of actuator force into the reduced translational dynamics chain.

The next notebook can then introduce environmental flow without reopening actuator, drag, gravity, or buoyancy validation.

The intended handoff is therefore:

```text
Notebook 05
Validated actuator-driven translational vehicle
                ↓
Notebook 06
Ocean currents and environmental flow effects
```

The engineering objective remains the same throughout HydroLab-3D:

> **Add one physical capability at a time, validate it independently, integrate it with the existing stack, and only then allow the next layer to depend on it.**

---

## Step 1 - Repository and Dependency Audit

Before implementing the thruster model, verify that Notebook 05 is operating against the validated HydroLab-3D production stack inherited from Notebooks 00–04.

This step does **not** implement thruster physics yet.

Its purpose is to confirm:

- the notebook is running from the expected HydroLab-3D repository;
- the production `src/` package is available;
- the validated kinematics module exists;
- the validated rigid-body translational dynamics module exists;
- the validated quadratic-drag module exists;
- the validated gravity/buoyancy module exists;
- the validated body-to-world transformation module exists;
- their required production functions can be imported successfully;
- the current state of the intended thruster production and test files is known before anything is created or modified.

The inherited production interfaces expected by Notebook 05 are:

```text
hydrolab3d.dynamics.kinematics
    integrate_position()

hydrolab3d.dynamics.rigid_body
    acceleration_from_force()
    integrate_velocity()

hydrolab3d.dynamics.drag
    quadratic_drag_force()

hydrolab3d.dynamics.buoyancy
    gravity_force()
    buoyancy_force()

hydrolab3d.utils.transforms
    rotation_body_to_world()
```

Notebook 05 will preserve the already validated explicit-Euler translational integration convention.

Position propagation:

$$
\mathbf{p}_{k+1}
=
\mathbf{p}_k
+
\mathbf{v}_k\Delta t
$$

Velocity propagation:

$$
\mathbf{v}_{k+1}
=
\mathbf{v}_k
+
\mathbf{a}_k\Delta t
$$

Notebook 05 will add actuator-generated thrust to the existing translational force balance:

$$
\mathbf{F}_{net,W}
=
\mathbf{F}_{T,W}
+
\mathbf{F}_{D,W}
+
\mathbf{F}_{G,W}
+
\mathbf{F}_{B,W}
$$

No inherited physics will be reimplemented in this step.

### Expected Result

This step passes if:

- all inherited production files exist;
- all required inherited functions import successfully;
- all required imported objects are callable;
- the intended Notebook 05 production targets can be inspected without modification.

The audit must report whether the following files already exist:

```text
src/hydrolab3d/dynamics/thrusters.py
tests/test_thrusters.py
```

No files should be created or modified during this step.

Only after this audit passes will Notebook 05 begin the notebook-local thruster prototype.

In [2]:
from pathlib import Path
import inspect
import sys


# ---------------------------------------------------------------------
# Resolve HydroLab-3D repository
# ---------------------------------------------------------------------

repo_root = Path.cwd().resolve()

if repo_root.name == "notebooks":
    repo_root = repo_root.parent

src_path = repo_root / "src"

if not src_path.exists():
    raise FileNotFoundError(
        f"HydroLab-3D source directory not found: {src_path}"
    )

src_path_str = str(src_path)

if src_path_str not in sys.path:
    sys.path.insert(0, src_path_str)


# ---------------------------------------------------------------------
# Expected inherited production files
# ---------------------------------------------------------------------

required_files = {
    "kinematics": (
        src_path
        / "hydrolab3d"
        / "dynamics"
        / "kinematics.py"
    ),
    "rigid_body": (
        src_path
        / "hydrolab3d"
        / "dynamics"
        / "rigid_body.py"
    ),
    "drag": (
        src_path
        / "hydrolab3d"
        / "dynamics"
        / "drag.py"
    ),
    "buoyancy": (
        src_path
        / "hydrolab3d"
        / "dynamics"
        / "buoyancy.py"
    ),
    "transforms": (
        src_path
        / "hydrolab3d"
        / "utils"
        / "transforms.py"
    ),
}


print("=" * 80)
print("HYDROLAB-3D NOTEBOOK 05 DEPENDENCY AUDIT")
print("=" * 80)

print(f"\nRepository root : {repo_root}")
print(f"Source path     : {src_path}")


# ---------------------------------------------------------------------
# Verify inherited production files
# ---------------------------------------------------------------------

print("\nInherited production files:")

for name, path in required_files.items():

    if not path.exists():
        raise FileNotFoundError(
            f"Required production file does not exist: {path}"
        )

    if not path.is_file():
        raise RuntimeError(
            f"Required production path is not a file: {path}"
        )

    if path.stat().st_size == 0:
        raise RuntimeError(
            f"Required production file is empty: {path}"
        )

    print(
        f"  [OK] {name:<12} "
        f"{path.relative_to(repo_root)}"
    )


# ---------------------------------------------------------------------
# Import validated production interfaces
# ---------------------------------------------------------------------

from hydrolab3d.dynamics.kinematics import integrate_position

from hydrolab3d.dynamics.rigid_body import (
    acceleration_from_force,
    integrate_velocity,
)

from hydrolab3d.dynamics.drag import quadratic_drag_force

from hydrolab3d.dynamics.buoyancy import (
    gravity_force,
    buoyancy_force,
)

from hydrolab3d.utils.transforms import rotation_body_to_world


inherited_functions = {
    "integrate_position": integrate_position,
    "acceleration_from_force": acceleration_from_force,
    "integrate_velocity": integrate_velocity,
    "quadratic_drag_force": quadratic_drag_force,
    "gravity_force": gravity_force,
    "buoyancy_force": buoyancy_force,
    "rotation_body_to_world": rotation_body_to_world,
}


# ---------------------------------------------------------------------
# Verify callable interfaces and inspect signatures
# ---------------------------------------------------------------------

print("\nInherited production interfaces:")

for name, function in inherited_functions.items():

    if not callable(function):
        raise TypeError(
            f"Expected callable production interface: {name}"
        )

    signature = inspect.signature(function)

    print(
        f"  [OK] {name}{signature}"
    )


# ---------------------------------------------------------------------
# Inspect intended Notebook 05 production targets
# ---------------------------------------------------------------------

thruster_path = (
    src_path
    / "hydrolab3d"
    / "dynamics"
    / "thrusters.py"
)

test_thruster_path = (
    repo_root
    / "tests"
    / "test_thrusters.py"
)


print("\nNotebook 05 production targets:")

print(
    "  thrusters.py      :",
    "EXISTS" if thruster_path.exists() else "not yet created",
)

print(
    "  test_thrusters.py :",
    "EXISTS" if test_thruster_path.exists() else "not yet created",
)


# ---------------------------------------------------------------------
# Final audit status
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 1 PASSED: inherited HydroLab-3D stack is available.")
print("No files were created or modified.")
print("=" * 80)

HYDROLAB-3D NOTEBOOK 05 DEPENDENCY AUDIT

Repository root : /home/agniacolyte/HydroLab-3D
Source path     : /home/agniacolyte/HydroLab-3D/src

Inherited production files:
  [OK] kinematics   src/hydrolab3d/dynamics/kinematics.py
  [OK] rigid_body   src/hydrolab3d/dynamics/rigid_body.py
  [OK] drag         src/hydrolab3d/dynamics/drag.py
  [OK] buoyancy     src/hydrolab3d/dynamics/buoyancy.py
  [OK] transforms   src/hydrolab3d/utils/transforms.py

Inherited production interfaces:
  [OK] integrate_position(position, velocity, dt)
  [OK] acceleration_from_force(force, mass)
  [OK] integrate_velocity(velocity, acceleration, dt)
  [OK] quadratic_drag_force(velocity, drag_coefficient)
  [OK] gravity_force(mass, gravity=9.81)
  [OK] buoyancy_force(fluid_density, displaced_volume, gravity=9.81)
  [OK] rotation_body_to_world(phi: float, theta: float, psi: float) -> numpy.ndarray

Notebook 05 production targets:
  thrusters.py      : EXISTS
  test_thrusters.py : EXISTS

STEP 1 PASSED: inherited 

## Step 2 - Inspect the Existing Thruster Implementation

The repository audit showed that both intended Notebook 05 production targets already exist:

```text
src/hydrolab3d/dynamics/thrusters.py
tests/test_thrusters.py
```

Before creating a notebook-local thruster prototype, we must determine what these existing files contain.

This is important because Notebook 05 must not accidentally:

- overwrite an existing implementation;
- duplicate already implemented behavior;
- inherit an undocumented API without examining it;
- assume that an existing thruster implementation is already validated;
- modify tests before understanding their current intent.

The HydroLab-3D development workflow is:

```text
inspect
   ↓
prototype locally
   ↓
validate
   ↓
promote reusable implementation
   ↓
write focused regression tests
   ↓
re-import production code
```

Existence of a production file does **not** by itself establish that its implementation is correct or that Notebook 05 should use it unchanged.

This step therefore performs a read-only source inspection.

### Questions This Audit Must Answer

For `thrusters.py`:

1. What functions or classes already exist?
2. What command convention is currently implemented?
3. Is command saturation already present?
4. How is maximum thrust represented?
5. How is thruster direction handled?
6. Is direction normalization performed, required, or absent?
7. Are body/world frame transformations handled here or elsewhere?
8. Is actuator lag or rate limiting already present?
9. What parameter validation exists?
10. Does the implementation introduce capabilities outside Notebook 05's reduced translational scope?

For `test_thrusters.py`:

1. What behavior is currently tested?
2. Are tests merely placeholders or substantive regression tests?
3. Do the tests establish assumptions that Notebook 05 should preserve?
4. Are important actuator properties currently untested?
5. Do any tests imply a different API from the reduced model intended here?

### Expected Result

This step passes when the complete current contents of both files can be inspected.

No file will be imported as trusted production logic yet.

No source file will be changed.

After inspection, the next step will decide whether the existing implementation should be:

- reused unchanged;
- revised after notebook-local validation;
- partially reused;
- or replaced by a smaller validated implementation.

That decision will be based on the actual code rather than the filenames alone.

In [3]:
from pathlib import Path


# ---------------------------------------------------------------------
# Existing Notebook 05 production targets
# ---------------------------------------------------------------------

thruster_path = (
    repo_root
    / "src"
    / "hydrolab3d"
    / "dynamics"
    / "thrusters.py"
)

test_thruster_path = (
    repo_root
    / "tests"
    / "test_thrusters.py"
)


# ---------------------------------------------------------------------
# Read-only validation
# ---------------------------------------------------------------------

for path in (thruster_path, test_thruster_path):

    if not path.exists():
        raise FileNotFoundError(
            f"Expected existing file was not found: {path}"
        )

    if not path.is_file():
        raise RuntimeError(
            f"Expected a file but found another path type: {path}"
        )


# ---------------------------------------------------------------------
# Display complete current contents
# ---------------------------------------------------------------------

print("=" * 80)
print("EXISTING THRUSTER PRODUCTION IMPLEMENTATION")
print("=" * 80)
print(f"Path: {thruster_path.relative_to(repo_root)}")
print("-" * 80)

thruster_source = thruster_path.read_text(encoding="utf-8")

if thruster_source.strip():
    print(thruster_source)
else:
    print("[EMPTY FILE]")


print("\n" + "=" * 80)
print("EXISTING THRUSTER REGRESSION TESTS")
print("=" * 80)
print(f"Path: {test_thruster_path.relative_to(repo_root)}")
print("-" * 80)

test_source = test_thruster_path.read_text(encoding="utf-8")

if test_source.strip():
    print(test_source)
else:
    print("[EMPTY FILE]")


print("\n" + "=" * 80)
print("STEP 2 INSPECTION COMPLETE")
print("No files were created or modified.")
print("=" * 80)

EXISTING THRUSTER PRODUCTION IMPLEMENTATION
Path: src/hydrolab3d/dynamics/thrusters.py
--------------------------------------------------------------------------------
[EMPTY FILE]

EXISTING THRUSTER REGRESSION TESTS
Path: tests/test_thrusters.py
--------------------------------------------------------------------------------
[EMPTY FILE]

STEP 2 INSPECTION COMPLETE
No files were created or modified.


## Step 3 - Notebook-Local Thruster Model and Command Mapping Validation

The repository inspection confirmed that:

```text
src/hydrolab3d/dynamics/thrusters.py
tests/test_thrusters.py
```

exist only as empty placeholders.

Notebook 05 can therefore define its actuator contract from first principles without preserving undocumented legacy behavior.

This step introduces the smallest notebook-local thruster model required by the reduced translational simulator.

No production files are modified yet.

---

### Reduced Thruster Model

The actuator receives a normalized scalar command:

$$
u \in \mathbb{R}
$$

The physically admissible command range is:

$$
u_{sat}
=
\operatorname{clip}(u,-1,1)
$$

For maximum thrust magnitude $F_{max}$:

$$
F_T
=
u_{sat}F_{max}
$$

The thruster is mounted along a fixed body-frame unit direction:

$$
\hat{\mathbf{d}}_B
=
\begin{bmatrix}
d_x \\
d_y \\
d_z
\end{bmatrix}
$$

with:

$$
\|\hat{\mathbf{d}}_B\|
=
1
$$

The resulting body-frame thrust vector is:

$$
\mathbf{F}_{T,B}
=
F_T\hat{\mathbf{d}}_B
$$

Therefore:

$$
\mathbf{F}_{T,B}
=
\operatorname{clip}(u,-1,1)
F_{max}
\hat{\mathbf{d}}_B
$$

---

### API Decision

The notebook-local prototype will use one function:

```text
thruster_force_body(command, max_thrust, direction_body)
```

returning a three-dimensional body-frame force vector.

The function will deliberately **not silently normalize** the supplied direction vector.

Instead, the API requires:

$$
\|\hat{\mathbf{d}}_B\|
\approx 1
$$

and rejects non-unit vectors.

This keeps configuration errors visible. A vector such as:

```text
[10, 0, 0]
```

must not accidentally produce a ten-times-larger effective actuator direction or be silently altered by hidden normalization.

A configured thruster direction is therefore treated as a physical unit-vector parameter.

---

### Parameter Contract

The notebook-local model requires:

#### Command

The command must be:

- scalar;
- finite.

Values outside:

$$
[-1,1]
$$

are permitted as requested commands but are physically saturated before thrust is generated.

#### Maximum Thrust

$F_{max}$ must be:

- scalar;
- finite;
- strictly positive.

A zero-thrust actuator could mathematically be represented, but it provides no useful actuator capability and is better represented by a zero command. The production contract will therefore require:

$$
F_{max}>0
$$

#### Thruster Direction

The body-frame direction must be:

- a NumPy-compatible vector with shape `(3,)`;
- finite;
- non-zero;
- unit length within numerical tolerance.

---

### Validation Cases

This step will validate the command-to-body-force mapping independently of vehicle dynamics.

For a forward-facing thruster:

$$
\hat{\mathbf{d}}_B
=
\begin{bmatrix}
1\\
0\\
0
\end{bmatrix}
$$

and:

$$
F_{max}=100\text{ N}
$$

the expected cases include:

| Command | Expected Body-Frame Thrust |
|---:|---|
| $0$ | $[0,0,0]$ N |
| $0.25$ | $[25,0,0]$ N |
| $0.5$ | $[50,0,0]$ N |
| $1$ | $[100,0,0]$ N |
| $2$ | $[100,0,0]$ N |
| $-0.5$ | $[-50,0,0]$ N |
| $-1$ | $[-100,0,0]$ N |
| $-2$ | $[-100,0,0]$ N |

This verifies:

- zero-command behavior;
- proportional unsaturated mapping;
- positive thrust;
- reverse thrust;
- positive saturation;
- negative saturation.

The step will also verify rejection of representative invalid inputs:

- non-scalar command;
- non-finite command;
- zero maximum thrust;
- negative maximum thrust;
- non-finite maximum thrust;
- incorrectly shaped direction vectors;
- zero direction;
- non-finite direction;
- non-unit direction.

---

### Scope of This Step

This step tests only:

```text
command
   ↓
saturation
   ↓
thrust magnitude
   ↓
body-frame thrust vector
```

It does **not** yet test:

- body-to-world transformation;
- vehicle orientation;
- drag;
- gravity;
- buoyancy;
- acceleration;
- velocity;
- position.

Those will be coupled only after the isolated actuator mapping passes.

---

### Expected Result

The step passes if:

1. all nominal command cases produce the analytically expected body-frame forces;
2. saturation occurs exactly at the configured maximum thrust;
3. negative commands correctly reverse force direction;
4. no unintended cross-axis force appears;
5. representative invalid parameters are rejected;
6. all returned nominal force vectors are finite three-dimensional arrays.

Only after this isolated actuator contract passes will the thruster be connected to the validated orientation infrastructure.

In [4]:
import numpy as np


# ---------------------------------------------------------------------
# Notebook-local reduced thruster prototype
# ---------------------------------------------------------------------

def thruster_force_body(command, max_thrust, direction_body):
    """
    Convert a normalized scalar thruster command into a body-frame
    thrust-force vector.

    Parameters
    ----------
    command : float
        Requested normalized command. Values outside [-1, 1] are
        saturated.

    max_thrust : float
        Maximum thrust magnitude in newtons. Must be finite and > 0.

    direction_body : array-like, shape (3,)
        Body-frame unit vector defining positive thrust direction.

    Returns
    -------
    np.ndarray, shape (3,)
        Body-frame thrust vector in newtons.
    """

    # Command validation
    if isinstance(command, (bool, np.bool_)) or not np.isscalar(command):
        raise ValueError("command must be a finite scalar.")

    command = float(command)

    if not np.isfinite(command):
        raise ValueError("command must be finite.")

    # Maximum-thrust validation
    if isinstance(max_thrust, (bool, np.bool_)) or not np.isscalar(max_thrust):
        raise ValueError("max_thrust must be a finite positive scalar.")

    max_thrust = float(max_thrust)

    if not np.isfinite(max_thrust) or max_thrust <= 0.0:
        raise ValueError("max_thrust must be finite and strictly positive.")

    # Direction validation
    direction_body = np.asarray(direction_body, dtype=float)

    if direction_body.shape != (3,):
        raise ValueError("direction_body must have shape (3,).")

    if not np.all(np.isfinite(direction_body)):
        raise ValueError("direction_body must contain only finite values.")

    direction_norm = np.linalg.norm(direction_body)

    if direction_norm == 0.0:
        raise ValueError("direction_body must not be the zero vector.")

    if not np.isclose(direction_norm, 1.0, rtol=1e-9, atol=1e-12):
        raise ValueError(
            "direction_body must be a unit vector; "
            f"received norm {direction_norm:.12g}."
        )

    # Saturated command-to-thrust mapping
    saturated_command = np.clip(command, -1.0, 1.0)

    thrust_magnitude = saturated_command * max_thrust

    return thrust_magnitude * direction_body


# ---------------------------------------------------------------------
# Nominal analytical validation
# ---------------------------------------------------------------------

max_thrust = 100.0

forward_body = np.array(
    [1.0, 0.0, 0.0],
    dtype=float,
)

nominal_cases = [
    (0.00, np.array([   0.0, 0.0, 0.0])),
    (0.25, np.array([  25.0, 0.0, 0.0])),
    (0.50, np.array([  50.0, 0.0, 0.0])),
    (1.00, np.array([ 100.0, 0.0, 0.0])),
    (2.00, np.array([ 100.0, 0.0, 0.0])),
    (-0.50, np.array([ -50.0, 0.0, 0.0])),
    (-1.00, np.array([-100.0, 0.0, 0.0])),
    (-2.00, np.array([-100.0, 0.0, 0.0])),
]


print("=" * 80)
print("THRUSTER COMMAND-TO-BODY-FORCE VALIDATION")
print("=" * 80)

for command, expected_force in nominal_cases:

    actual_force = thruster_force_body(
        command=command,
        max_thrust=max_thrust,
        direction_body=forward_body,
    )

    np.testing.assert_allclose(
        actual_force,
        expected_force,
        rtol=0.0,
        atol=1e-12,
    )

    assert actual_force.shape == (3,)
    assert np.all(np.isfinite(actual_force))

    print(
        f"command={command:>5.2f}  "
        f"expected={expected_force}  "
        f"actual={actual_force}  "
        f"[PASS]"
    )


# ---------------------------------------------------------------------
# Additional direction sanity check
# ---------------------------------------------------------------------

vertical_body = np.array(
    [0.0, 0.0, 1.0],
    dtype=float,
)

vertical_force = thruster_force_body(
    command=0.4,
    max_thrust=50.0,
    direction_body=vertical_body,
)

np.testing.assert_allclose(
    vertical_force,
    np.array([0.0, 0.0, 20.0]),
    rtol=0.0,
    atol=1e-12,
)

print(
    "\nDirection sanity check:",
    vertical_force,
    "[PASS]",
)


# ---------------------------------------------------------------------
# Invalid-input validation
# ---------------------------------------------------------------------

invalid_cases = [
    (
        "non-scalar command",
        dict(
            command=[0.5],
            max_thrust=100.0,
            direction_body=forward_body,
        ),
    ),
    (
        "non-finite command",
        dict(
            command=np.nan,
            max_thrust=100.0,
            direction_body=forward_body,
        ),
    ),
    (
        "zero max thrust",
        dict(
            command=0.5,
            max_thrust=0.0,
            direction_body=forward_body,
        ),
    ),
    (
        "negative max thrust",
        dict(
            command=0.5,
            max_thrust=-100.0,
            direction_body=forward_body,
        ),
    ),
    (
        "non-finite max thrust",
        dict(
            command=0.5,
            max_thrust=np.inf,
            direction_body=forward_body,
        ),
    ),
    (
        "wrong direction shape",
        dict(
            command=0.5,
            max_thrust=100.0,
            direction_body=[1.0, 0.0],
        ),
    ),
    (
        "zero direction",
        dict(
            command=0.5,
            max_thrust=100.0,
            direction_body=[0.0, 0.0, 0.0],
        ),
    ),
    (
        "non-finite direction",
        dict(
            command=0.5,
            max_thrust=100.0,
            direction_body=[1.0, np.nan, 0.0],
        ),
    ),
    (
        "non-unit direction",
        dict(
            command=0.5,
            max_thrust=100.0,
            direction_body=[2.0, 0.0, 0.0],
        ),
    ),
]


print("\n" + "=" * 80)
print("INVALID-PARAMETER VALIDATION")
print("=" * 80)

for case_name, kwargs in invalid_cases:

    try:
        thruster_force_body(**kwargs)

    except ValueError as exc:
        print(
            f"{case_name:<25} "
            f"[PASS] rejected -> {exc}"
        )

    else:
        raise AssertionError(
            f"Invalid case was not rejected: {case_name}"
        )


print("\n" + "=" * 80)
print("STEP 3 PASSED")
print("Notebook-local command-to-body-thrust model validated.")
print("No production files were modified.")
print("=" * 80)

THRUSTER COMMAND-TO-BODY-FORCE VALIDATION
command= 0.00  expected=[0. 0. 0.]  actual=[0. 0. 0.]  [PASS]
command= 0.25  expected=[25.  0.  0.]  actual=[25.  0.  0.]  [PASS]
command= 0.50  expected=[50.  0.  0.]  actual=[50.  0.  0.]  [PASS]
command= 1.00  expected=[100.   0.   0.]  actual=[100.   0.   0.]  [PASS]
command= 2.00  expected=[100.   0.   0.]  actual=[100.   0.   0.]  [PASS]
command=-0.50  expected=[-50.   0.   0.]  actual=[-50.  -0.  -0.]  [PASS]
command=-1.00  expected=[-100.    0.    0.]  actual=[-100.   -0.   -0.]  [PASS]
command=-2.00  expected=[-100.    0.    0.]  actual=[-100.   -0.   -0.]  [PASS]

Direction sanity check: [ 0.  0. 20.] [PASS]

INVALID-PARAMETER VALIDATION
non-scalar command        [PASS] rejected -> command must be a finite scalar.
non-finite command        [PASS] rejected -> command must be finite.
zero max thrust           [PASS] rejected -> max_thrust must be finite and strictly positive.
negative max thrust       [PASS] rejected -> max_thrust must 

## Step 4 - Thruster Direction and Body-to-World Transformation Validation

The notebook-local thruster model has now been validated in the vehicle body frame.

The next requirement is to verify that a thruster physically attached to the vehicle rotates correctly with vehicle orientation.

A thruster direction is defined in the body frame:

$$
\hat{\mathbf{d}}_B
$$

and produces body-frame thrust:

$$
\mathbf{F}_{T,B}
=
F_T\hat{\mathbf{d}}_B
$$

The corresponding world-frame thrust must be obtained using the already validated HydroLab-3D body-to-world rotation:

$$
\mathbf{F}_{T,W}
=
R_B^W
\mathbf{F}_{T,B}
$$

where:

$$
R_B^W
=
R_z(\psi)R_y(\theta)R_x(\phi)
$$

This step does **not** create a second orientation implementation.

It reuses:

```text
rotation_body_to_world()
```

from:

```text
src/hydrolab3d/utils/transforms.py
```

---

### Physical Interpretation

Consider a forward-facing thruster:

$$
\hat{\mathbf{d}}_B
=
\begin{bmatrix}
1\\
0\\
0
\end{bmatrix}
$$

At zero orientation:

$$
\phi=\theta=\psi=0
$$

the rotation matrix is the identity:

$$
R_B^W=I
$$

and therefore forward body thrust remains forward world thrust:

$$
\mathbf{F}_{T,W}
=
\begin{bmatrix}
F_T\\
0\\
0
\end{bmatrix}
$$

For a positive yaw rotation of:

$$
\psi=90^\circ
$$

the positive body $x$ axis should rotate onto positive world $y$:

$$
\begin{bmatrix}
F_T\\
0\\
0
\end{bmatrix}_B
\rightarrow
\begin{bmatrix}
0\\
F_T\\
0
\end{bmatrix}_W
$$

For a positive pitch rotation of:

$$
\theta=90^\circ
$$

under the established HydroLab-3D convention, positive body $x$ rotates toward negative world $z$:

$$
\begin{bmatrix}
F_T\\
0\\
0
\end{bmatrix}_B
\rightarrow
\begin{bmatrix}
0\\
0\\
-F_T
\end{bmatrix}_W
$$

Roll does not alter a thrust vector lying exactly along the body $x$ axis, because roll is rotation about that same axis.

---

### Validation Strategy

This step will test four properties.

#### 1. Identity Orientation

Zero roll, pitch, and yaw must leave the thrust vector unchanged.

#### 2. Known Yaw Rotation

A $90^\circ$ positive yaw must rotate positive body $x$ thrust into positive world $y$.

#### 3. Known Pitch Rotation

A $90^\circ$ positive pitch must rotate positive body $x$ thrust into negative world $z$ under the validated rotation convention.

#### 4. Magnitude Preservation

Rotation must change only the direction of thrust.

For every valid rotation:

$$
\|\mathbf{F}_{T,W}\|
=
\|\mathbf{F}_{T,B}\|
$$

This property is important because an orientation transform must never create or destroy actuator force.

---

### Additional Arbitrary-Orientation Check

A nontrivial roll-pitch-yaw combination will also be tested numerically.

Rather than hard-coding another independently derived transformation, the validation will confirm that:

- the transformed vector remains finite;
- its magnitude is preserved;
- applying the transpose of the rotation matrix recovers the original body-frame vector.

Because a valid rotation matrix is orthogonal:

$$
(R_B^W)^T R_B^W
=
I
$$

and therefore:

$$
\mathbf{F}_{T,B}
=
(R_B^W)^T
\mathbf{F}_{T,W}
$$

This provides an additional consistency check without duplicating the production rotation mathematics.

---

### Scope of This Step

This step validates:

```text
command
   ↓
body-frame thrust
   ↓
validated rotation matrix
   ↓
world-frame thrust
```

It still does **not** introduce:

- drag;
- gravity;
- buoyancy;
- net force;
- acceleration;
- velocity integration;
- position integration.

Those enter only after the actuator-frame chain has been validated independently.

---

### Expected Result

The step passes if:

- zero orientation preserves the thrust vector;
- known yaw produces the expected world-axis direction;
- known pitch produces the expected world-axis direction;
- roll about the thrust axis preserves that thrust vector;
- arbitrary orientation preserves thrust magnitude;
- inverse rotation recovers the original body-frame thrust;
- no non-finite values appear.

After this passes, the actuator force can be connected to the complete validated HydroLab-3D translational force balance.

In [5]:
import numpy as np


# ---------------------------------------------------------------------
# Helper: transform notebook-local thrust into world frame
# ---------------------------------------------------------------------

def thruster_force_world(
    command,
    max_thrust,
    direction_body,
    phi,
    theta,
    psi,
):
    """
    Compute world-frame thruster force by combining the validated
    notebook-local body-frame thruster model with the existing
    HydroLab-3D body-to-world rotation.
    """

    force_body = thruster_force_body(
        command=command,
        max_thrust=max_thrust,
        direction_body=direction_body,
    )

    rotation = rotation_body_to_world(
        phi=phi,
        theta=theta,
        psi=psi,
    )

    force_world = rotation @ force_body

    return force_world


# ---------------------------------------------------------------------
# Common actuator configuration
# ---------------------------------------------------------------------

command = 0.5
max_thrust = 100.0

forward_body = np.array(
    [1.0, 0.0, 0.0],
    dtype=float,
)

expected_magnitude = 50.0


print("=" * 80)
print("THRUSTER BODY-TO-WORLD TRANSFORMATION VALIDATION")
print("=" * 80)


# ---------------------------------------------------------------------
# Case 1: identity orientation
# ---------------------------------------------------------------------

force_identity = thruster_force_world(
    command=command,
    max_thrust=max_thrust,
    direction_body=forward_body,
    phi=0.0,
    theta=0.0,
    psi=0.0,
)

expected_identity = np.array(
    [50.0, 0.0, 0.0]
)

np.testing.assert_allclose(
    force_identity,
    expected_identity,
    rtol=0.0,
    atol=1e-12,
)

print(
    "\nIdentity orientation:"
    f"\n  expected = {expected_identity}"
    f"\n  actual   = {force_identity}"
    "\n  [PASS]"
)


# ---------------------------------------------------------------------
# Case 2: +90 deg yaw
# ---------------------------------------------------------------------

force_yaw_90 = thruster_force_world(
    command=command,
    max_thrust=max_thrust,
    direction_body=forward_body,
    phi=0.0,
    theta=0.0,
    psi=np.pi / 2.0,
)

expected_yaw_90 = np.array(
    [0.0, 50.0, 0.0]
)

np.testing.assert_allclose(
    force_yaw_90,
    expected_yaw_90,
    rtol=0.0,
    atol=1e-12,
)

print(
    "\n+90 deg yaw:"
    f"\n  expected = {expected_yaw_90}"
    f"\n  actual   = {force_yaw_90}"
    "\n  [PASS]"
)


# ---------------------------------------------------------------------
# Case 3: +90 deg pitch
# ---------------------------------------------------------------------

force_pitch_90 = thruster_force_world(
    command=command,
    max_thrust=max_thrust,
    direction_body=forward_body,
    phi=0.0,
    theta=np.pi / 2.0,
    psi=0.0,
)

expected_pitch_90 = np.array(
    [0.0, 0.0, -50.0]
)

np.testing.assert_allclose(
    force_pitch_90,
    expected_pitch_90,
    rtol=0.0,
    atol=1e-12,
)

print(
    "\n+90 deg pitch:"
    f"\n  expected = {expected_pitch_90}"
    f"\n  actual   = {force_pitch_90}"
    "\n  [PASS]"
)


# ---------------------------------------------------------------------
# Case 4: roll about the body x thrust axis
# ---------------------------------------------------------------------

force_roll = thruster_force_world(
    command=command,
    max_thrust=max_thrust,
    direction_body=forward_body,
    phi=np.deg2rad(63.0),
    theta=0.0,
    psi=0.0,
)

expected_roll = np.array(
    [50.0, 0.0, 0.0]
)

np.testing.assert_allclose(
    force_roll,
    expected_roll,
    rtol=0.0,
    atol=1e-12,
)

print(
    "\nRoll about thrust axis:"
    f"\n  expected = {expected_roll}"
    f"\n  actual   = {force_roll}"
    "\n  [PASS]"
)


# ---------------------------------------------------------------------
# Case 5: arbitrary orientation
# ---------------------------------------------------------------------

phi = np.deg2rad(20.0)
theta = np.deg2rad(-35.0)
psi = np.deg2rad(70.0)

force_body = thruster_force_body(
    command=command,
    max_thrust=max_thrust,
    direction_body=forward_body,
)

rotation = rotation_body_to_world(
    phi=phi,
    theta=theta,
    psi=psi,
)

force_world = rotation @ force_body


# Magnitude preservation
np.testing.assert_allclose(
    np.linalg.norm(force_world),
    np.linalg.norm(force_body),
    rtol=1e-12,
    atol=1e-12,
)


# Orthogonality
np.testing.assert_allclose(
    rotation.T @ rotation,
    np.eye(3),
    rtol=1e-12,
    atol=1e-12,
)


# Recover original body-frame force
recovered_body = rotation.T @ force_world

np.testing.assert_allclose(
    recovered_body,
    force_body,
    rtol=1e-12,
    atol=1e-12,
)


assert np.all(np.isfinite(force_world))


print(
    "\nArbitrary orientation:"
    f"\n  body force      = {force_body}"
    f"\n  world force     = {force_world}"
    f"\n  body magnitude  = {np.linalg.norm(force_body):.12f} N"
    f"\n  world magnitude = {np.linalg.norm(force_world):.12f} N"
    f"\n  recovered body  = {recovered_body}"
    "\n  [PASS]"
)


# ---------------------------------------------------------------------
# Final status
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 4 PASSED")
print("Thruster body-to-world direction behavior validated.")
print("Rotation preserves thrust magnitude and introduces no spurious force.")
print("=" * 80)

THRUSTER BODY-TO-WORLD TRANSFORMATION VALIDATION

Identity orientation:
  expected = [50.  0.  0.]
  actual   = [50.  0.  0.]
  [PASS]

+90 deg yaw:
  expected = [ 0. 50.  0.]
  actual   = [3.061617e-15 5.000000e+01 0.000000e+00]
  [PASS]

+90 deg pitch:
  expected = [  0.   0. -50.]
  actual   = [ 3.061617e-15  0.000000e+00 -5.000000e+01]
  [PASS]

Roll about thrust axis:
  expected = [50.  0.  0.]
  actual   = [50.  0.  0.]
  [PASS]

Arbitrary orientation:
  body force      = [50.  0.  0.]
  world force     = [14.00832498 38.48755657 28.67882182]
  body magnitude  = 50.000000000000 N
  world magnitude = 50.000000000000 N
  recovered body  = [5.00000000e+01 0.00000000e+00 3.55271368e-15]
  [PASS]

STEP 4 PASSED
Thruster body-to-world direction behavior validated.
Rotation preserves thrust magnitude and introduces no spurious force.


## Step 5 - Integrate Thruster Force with the Validated Translational Dynamics

The actuator mapping and reference-frame behavior have now passed independently.

This step connects the notebook-local thruster model to the complete translational HydroLab-3D force balance inherited from Notebooks 00–04.

The governing equation becomes:

$$
m\dot{\mathbf{v}}
=
\mathbf{F}_T
+
\mathbf{F}_D
+
\mathbf{F}_G
+
\mathbf{F}_B
$$

with:

$$
\mathbf{F}_D
=
-C_D
\|\mathbf{v}\|
\mathbf{v}
$$

$$
\mathbf{F}_G
=
\begin{bmatrix}
0\\
0\\
-mg
\end{bmatrix}
$$

and:

$$
\mathbf{F}_B
=
\begin{bmatrix}
0\\
0\\
\rho gV
\end{bmatrix}
$$

No inherited force model is reimplemented here.

The step uses the already validated production functions:

```text
quadratic_drag_force()
gravity_force()
buoyancy_force()
acceleration_from_force()
integrate_velocity()
integrate_position()
```

together with the notebook-local:

```text
thruster_force_world()
```

---

### Numerical Ordering

The integration order remains the explicit-Euler convention inherited from the previous HydroLab-3D notebooks.

At each timestep:

1. evaluate thrust from the current command;
2. evaluate drag from the current velocity;
3. evaluate gravity and buoyancy;
4. construct net force;
5. compute acceleration;
6. propagate position using the beginning-of-step velocity;
7. propagate velocity using the current acceleration.

Thus:

$$
\mathbf{F}_{net,k}
=
\mathbf{F}_{T,k}
+
\mathbf{F}_{D,k}
+
\mathbf{F}_{G}
+
\mathbf{F}_{B}
$$

$$
\mathbf{a}_k
=
\frac{\mathbf{F}_{net,k}}{m}
$$

$$
\mathbf{p}_{k+1}
=
\mathbf{p}_k
+
\mathbf{v}_k\Delta t
$$

$$
\mathbf{v}_{k+1}
=
\mathbf{v}_k
+
\mathbf{a}_k\Delta t
$$

This step deliberately preserves that established numerical contract rather than changing integration schemes while introducing a new physical component.

---

### Canonical Vehicle

Use the same convenient vehicle scale established in Notebook 04:

$$
m=20\text{ kg}
$$

$$
\rho=1000\text{ kg/m}^3
$$

with displaced volume:

$$
V
=
\frac{m}{\rho}
=
0.020\text{ m}^3
$$

Therefore:

$$
\rho V=m
$$

and the vehicle is neutrally buoyant:

$$
\mathbf{F}_G+\mathbf{F}_B
=
\mathbf{0}
$$

Neutral buoyancy isolates the horizontal thruster/drag dynamics while still keeping gravity and buoyancy explicitly inside the complete force balance.

---

### Thruster Configuration

Use a forward body-axis thruster:

$$
\hat{\mathbf{d}}_B
=
\begin{bmatrix}
1\\
0\\
0
\end{bmatrix}
$$

with:

$$
F_{max}=40\text{ N}
$$

and constant command:

$$
u=0.5
$$

The commanded thrust is therefore:

$$
F_T
=
0.5(40)
=
20\text{ N}
$$

The vehicle orientation remains zero:

$$
\phi=\theta=\psi=0
$$

so:

$$
\mathbf{F}_{T,W}
=
\begin{bmatrix}
20\\
0\\
0
\end{bmatrix}
\text{ N}
$$

---

### Analytical Terminal Speed

Using the validated quadratic drag model with:

$$
C_D=5
$$

one-axis positive motion gives:

$$
F_D
=
-C_Dv_x^2
$$

At terminal speed:

$$
F_T+F_D=0
$$

therefore:

$$
F_T
=
C_Dv_{x,\infty}^2
$$

and:

$$
v_{x,\infty}
=
\sqrt{\frac{F_T}{C_D}}
$$

For:

$$
F_T=20\text{ N}
$$

and:

$$
C_D=5
$$

the analytical prediction is:

$$
v_{x,\infty}
=
\sqrt{\frac{20}{5}}
=
2\text{ m/s}
$$

The numerical simulation should converge closely toward this value.

---

### Validation Cases

This step performs two end-to-end experiments.

#### Case A - Zero Command

For a neutrally buoyant vehicle initially at rest:

$$
u=0
$$

the force balance should satisfy:

$$
\mathbf{F}_T
=
\mathbf{F}_D
=
\mathbf{F}_G+\mathbf{F}_B
=
\mathbf{0}
$$

and therefore:

$$
\mathbf{a}
=
\mathbf{v}
=
\mathbf{p}
=
\mathbf{0}
$$

throughout the simulation.

This verifies that simply adding the actuator layer does not introduce spontaneous vehicle motion.

#### Case B - Constant Forward Thrust

For:

$$
u=0.5
$$

the vehicle should:

- initially accelerate along positive world $x$;
- develop increasing opposing drag;
- experience decreasing net forward force;
- asymptotically approach the analytical terminal speed;
- exhibit no unintended $y$ or $z$ motion.

At equilibrium:

$$
F_{T,x}
+
F_{D,x}
\approx
0
$$

and:

$$
a_x
\approx
0
$$

---

### Success Criteria

This step passes if:

- the zero-command neutrally buoyant vehicle remains stationary;
- constant positive thrust produces positive $x$ motion;
- no unintended $y$ or $z$ velocity or displacement develops;
- thrust remains exactly at its expected commanded value;
- drag opposes forward motion;
- gravity and buoyancy cancel for the neutral vehicle;
- all states and forces remain finite;
- the numerical final speed agrees closely with:

$$
v_{x,\infty}=2\text{ m/s}
$$

- the final thrust-drag force balance is approximately zero.

Passing this experiment establishes the complete reduced translational chain:

```text
command
   ↓
thruster
   ↓
world-frame thrust
   ↓
thrust + drag + gravity + buoyancy
   ↓
net force
   ↓
acceleration
   ↓
velocity
   ↓
position
```

In [6]:
import numpy as np


# ---------------------------------------------------------------------
# Canonical vehicle and environment parameters
# ---------------------------------------------------------------------

mass = 20.0
fluid_density = 1000.0
displaced_volume = mass / fluid_density
gravity = 9.81

drag_coefficient = 5.0

max_thrust = 40.0
command_active = 0.5

direction_body = np.array(
    [1.0, 0.0, 0.0],
    dtype=float,
)

phi = 0.0
theta = 0.0
psi = 0.0

dt = 0.01
duration = 20.0
steps = int(duration / dt)


# ---------------------------------------------------------------------
# Constant gravity and buoyancy
# ---------------------------------------------------------------------

force_gravity = gravity_force(
    mass,
    gravity,
)

force_buoyancy = buoyancy_force(
    fluid_density,
    displaced_volume,
    gravity,
)

np.testing.assert_allclose(
    force_gravity + force_buoyancy,
    np.zeros(3),
    rtol=0.0,
    atol=1e-12,
)


# ---------------------------------------------------------------------
# Small simulation helper using the inherited integration convention
# ---------------------------------------------------------------------

def simulate_thruster_case(command):
    position = np.zeros(3, dtype=float)
    velocity = np.zeros(3, dtype=float)

    history = []

    for step in range(steps + 1):
        time = step * dt

        force_thrust = thruster_force_world(
            command=command,
            max_thrust=max_thrust,
            direction_body=direction_body,
            phi=phi,
            theta=theta,
            psi=psi,
        )

        force_drag = quadratic_drag_force(
            velocity,
            drag_coefficient,
        )

        force_net = (
            force_thrust
            + force_drag
            + force_gravity
            + force_buoyancy
        )

        acceleration = acceleration_from_force(
            force_net,
            mass,
        )

        history.append(
            {
                "time": time,
                "position": position.copy(),
                "velocity": velocity.copy(),
                "thrust": force_thrust.copy(),
                "drag": force_drag.copy(),
                "gravity": force_gravity.copy(),
                "buoyancy": force_buoyancy.copy(),
                "net_force": force_net.copy(),
                "acceleration": acceleration.copy(),
            }
        )

        if step == steps:
            break

        # Preserve the validated HydroLab-3D explicit-Euler ordering:
        # position uses beginning-of-step velocity.
        position = integrate_position(
            position,
            velocity,
            dt,
        )

        velocity = integrate_velocity(
            velocity,
            acceleration,
            dt,
        )

    return history


# =====================================================================
# CASE A - ZERO COMMAND / NEUTRAL BUOYANCY
# =====================================================================

zero_history = simulate_thruster_case(
    command=0.0,
)

zero_final = zero_history[-1]

np.testing.assert_allclose(
    zero_final["position"],
    np.zeros(3),
    rtol=0.0,
    atol=1e-12,
)

np.testing.assert_allclose(
    zero_final["velocity"],
    np.zeros(3),
    rtol=0.0,
    atol=1e-12,
)

np.testing.assert_allclose(
    zero_final["net_force"],
    np.zeros(3),
    rtol=0.0,
    atol=1e-12,
)


print("=" * 80)
print("CASE A - ZERO COMMAND / NEUTRAL BUOYANCY")
print("=" * 80)

print(
    f"Final position : {zero_final['position']}"
)

print(
    f"Final velocity : {zero_final['velocity']}"
)

print(
    f"Final net force: {zero_final['net_force']}"
)

print("[PASS] Vehicle remained stationary.")


# =====================================================================
# CASE B - CONSTANT FORWARD THRUST
# =====================================================================

active_history = simulate_thruster_case(
    command=command_active,
)

active_final = active_history[-1]


# ---------------------------------------------------------------------
# Analytical prediction
# ---------------------------------------------------------------------

expected_thrust = command_active * max_thrust

terminal_speed_analytic = np.sqrt(
    expected_thrust / drag_coefficient
)

terminal_speed_numeric = active_final[
    "velocity"
][0]

terminal_speed_error = abs(
    terminal_speed_numeric
    - terminal_speed_analytic
)


# ---------------------------------------------------------------------
# End-to-end assertions
# ---------------------------------------------------------------------

np.testing.assert_allclose(
    active_final["thrust"],
    np.array([expected_thrust, 0.0, 0.0]),
    rtol=0.0,
    atol=1e-12,
)

assert active_final["velocity"][0] > 0.0
assert active_final["position"][0] > 0.0

np.testing.assert_allclose(
    active_final["velocity"][1:],
    np.zeros(2),
    rtol=0.0,
    atol=1e-12,
)

np.testing.assert_allclose(
    active_final["position"][1:],
    np.zeros(2),
    rtol=0.0,
    atol=1e-12,
)

assert active_final["drag"][0] < 0.0

np.testing.assert_allclose(
    force_gravity + force_buoyancy,
    np.zeros(3),
    rtol=0.0,
    atol=1e-12,
)

assert terminal_speed_error < 1e-6

assert abs(
    active_final["net_force"][0]
) < 1e-5


# ---------------------------------------------------------------------
# Finite-value audit across complete simulation
# ---------------------------------------------------------------------

for record in active_history:
    for name in (
        "position",
        "velocity",
        "thrust",
        "drag",
        "gravity",
        "buoyancy",
        "net_force",
        "acceleration",
    ):
        assert np.all(
            np.isfinite(record[name])
        ), (
            f"Non-finite {name} at "
            f"t={record['time']}"
        )


# ---------------------------------------------------------------------
# Results
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("CASE B - CONSTANT FORWARD THRUST")
print("=" * 80)

print(
    f"Command                  : {command_active:.6f}"
)

print(
    f"Maximum thrust           : {max_thrust:.6f} N"
)

print(
    f"Applied thrust           : {expected_thrust:.6f} N"
)

print(
    f"Drag coefficient         : {drag_coefficient:.6f}"
)

print(
    f"Analytical terminal speed: "
    f"{terminal_speed_analytic:.12f} m/s"
)

print(
    f"Numerical final speed    : "
    f"{terminal_speed_numeric:.12f} m/s"
)

print(
    f"Absolute speed error     : "
    f"{terminal_speed_error:.12e} m/s"
)

print(
    f"Final position           : "
    f"{active_final['position']}"
)

print(
    f"Final thrust             : "
    f"{active_final['thrust']}"
)

print(
    f"Final drag               : "
    f"{active_final['drag']}"
)

print(
    f"Final gravity            : "
    f"{active_final['gravity']}"
)

print(
    f"Final buoyancy           : "
    f"{active_final['buoyancy']}"
)

print(
    f"Final net force          : "
    f"{active_final['net_force']}"
)

print(
    f"Final acceleration       : "
    f"{active_final['acceleration']}"
)


print("\n" + "=" * 80)
print("STEP 5 PASSED")
print("Complete actuator-driven translational dynamics validated.")
print("Numerical terminal speed agrees with the analytical prediction.")
print("No unintended cross-axis motion was observed.")
print("=" * 80)

CASE A - ZERO COMMAND / NEUTRAL BUOYANCY
Final position : [0. 0. 0.]
Final velocity : [0. 0. 0.]
Final net force: [0. 0. 0.]
[PASS] Vehicle remained stationary.

CASE B - CONSTANT FORWARD THRUST
Command                  : 0.500000
Maximum thrust           : 40.000000 N
Applied thrust           : 20.000000 N
Drag coefficient         : 5.000000
Analytical terminal speed: 2.000000000000 m/s
Numerical final speed    : 1.999999992493 m/s
Absolute speed error     : 7.507332311363e-09 m/s
Final position           : [37.22353811  0.          0.        ]
Final thrust             : [20.  0.  0.]
Final drag               : [-19.99999985  -0.          -0.        ]
Final gravity            : [   0.     0.  -196.2]
Final buoyancy           : [  0.    0.  196.2]
Final net force          : [1.50146647e-07 0.00000000e+00 0.00000000e+00]
Final acceleration       : [7.50733236e-09 0.00000000e+00 0.00000000e+00]

STEP 5 PASSED
Complete actuator-driven translational dynamics validated.
Numerical terminal s

## Step 6 - Generate Persistent Validation Artifacts

The reduced thruster model has now passed its core physical validation.

The validated chain is:

```text
command
   ↓
command saturation
   ↓
body-frame thrust
   ↓
body-to-world transformation
   ↓
thrust + drag + gravity + buoyancy
   ↓
net force
   ↓
acceleration
   ↓
velocity
   ↓
position
```

The next task is to preserve the canonical experiment as reproducible engineering evidence.

This step creates only persistent artifacts that demonstrate already validated behavior.

It does **not** introduce new physics.

---

### Output Directory

Notebook 05 owns:

```text
results/notebooks/05_thruster_model/
```

This step will create:

```text
results/notebooks/05_thruster_model/
├── data/
│   └── thruster_validation.csv
└── plots/
    ├── command_thrust_response.png
    ├── velocity_validation.png
    └── force_balance.png
```

The force-balance plot is retained because it demonstrates a distinct physical result that the velocity plot alone does not show:

$$
F_T + F_D \rightarrow 0
$$

as the vehicle approaches terminal speed.

---

### Command-Thrust Response

The first figure will evaluate the static actuator map:

$$
F_T
=
\operatorname{clip}(u,-1,1)F_{max}
$$

over commands extending beyond the physical actuator range.

The plot should clearly show:

- linear behavior for:

$$
-1 \le u \le 1
$$

- saturation for:

$$
u<-1
$$

and:

$$
u>1
$$

For the validated actuator:

$$
F_{max}=40\text{ N}
$$

so thrust must remain bounded by:

$$
-40\text{ N}
\le
F_T
\le
40\text{ N}
$$

---

### Velocity Validation Figure

The second figure will preserve the canonical constant-command experiment from Step 5.

For:

$$
u=0.5
$$

$$
F_T=20\text{ N}
$$

$$
C_D=5
$$

the analytical terminal speed is:

$$
v_{x,\infty}=2\text{ m/s}
$$

The numerical velocity history should asymptotically approach this analytical reference.

This figure therefore provides persistent evidence that the actuator integrates correctly with quadratic drag and translational dynamics.

---

### Force-Balance Figure

The third figure will show the horizontal forces:

- thruster force;
- drag force;
- net force.

Initially:

$$
F_D=0
$$

so:

$$
F_{net,x}=F_T
$$

As velocity increases, drag magnitude increases.

At equilibrium:

$$
F_{T,x}
+
F_{D,x}
\approx
0
$$

and therefore:

$$
F_{net,x}
\approx
0
$$

This directly exposes the mechanism responsible for terminal velocity.

---

### Validation CSV

The canonical constant-thrust experiment will be stored as:

```text
thruster_validation.csv
```

with one row per simulation sample.

The dataset will contain:

```text
time

command

thrust_x
thrust_y
thrust_z

drag_x
drag_y
drag_z

gravity_x
gravity_y
gravity_z

buoyancy_x
buoyancy_y
buoyancy_z

net_force_x
net_force_y
net_force_z

acceleration_x
acceleration_y
acceleration_z

velocity_x
velocity_y
velocity_z

position_x
position_y
position_z
```

This preserves enough information to reconstruct the complete canonical translational experiment without rerunning the notebook.

---

### Artifact Integrity

After writing the outputs, this step will verify that:

- every expected file exists;
- every expected file is non-empty;
- the CSV row count matches the simulation history;
- the final CSV state matches the validated Step 5 final state;
- the final recorded velocity remains consistent with the analytical terminal-speed result.

The figures are scientific validation artifacts, not decorative visualizations.

---

### Expected Result

This step passes when the Notebook 05 validation directory contains:

```text
data/thruster_validation.csv

plots/command_thrust_response.png

plots/velocity_validation.png

plots/force_balance.png
```

and all artifacts pass the integrity checks.

After this step, the notebook-local implementation and its canonical validation evidence are complete.

The next engineering stage can then promote the validated actuator logic into the HydroLab-3D production package.

In [7]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


# ---------------------------------------------------------------------
# Notebook 05 output directories
# ---------------------------------------------------------------------

output_root = (
    repo_root
    / "results"
    / "notebooks"
    / "05_thruster_model"
)

data_dir = output_root / "data"
plots_dir = output_root / "plots"

data_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)


csv_path = data_dir / "thruster_validation.csv"

command_plot_path = (
    plots_dir
    / "command_thrust_response.png"
)

velocity_plot_path = (
    plots_dir
    / "velocity_validation.png"
)

force_plot_path = (
    plots_dir
    / "force_balance.png"
)


# ---------------------------------------------------------------------
# Convert validated Step 5 history into arrays
# ---------------------------------------------------------------------

times = np.array([
    record["time"]
    for record in active_history
])

positions = np.vstack([
    record["position"]
    for record in active_history
])

velocities = np.vstack([
    record["velocity"]
    for record in active_history
])

thrusts = np.vstack([
    record["thrust"]
    for record in active_history
])

drags = np.vstack([
    record["drag"]
    for record in active_history
])

gravities = np.vstack([
    record["gravity"]
    for record in active_history
])

buoyancies = np.vstack([
    record["buoyancy"]
    for record in active_history
])

net_forces = np.vstack([
    record["net_force"]
    for record in active_history
])

accelerations = np.vstack([
    record["acceleration"]
    for record in active_history
])


# ---------------------------------------------------------------------
# Write canonical validation CSV
# ---------------------------------------------------------------------

fieldnames = [
    "time",
    "command",

    "thrust_x",
    "thrust_y",
    "thrust_z",

    "drag_x",
    "drag_y",
    "drag_z",

    "gravity_x",
    "gravity_y",
    "gravity_z",

    "buoyancy_x",
    "buoyancy_y",
    "buoyancy_z",

    "net_force_x",
    "net_force_y",
    "net_force_z",

    "acceleration_x",
    "acceleration_y",
    "acceleration_z",

    "velocity_x",
    "velocity_y",
    "velocity_z",

    "position_x",
    "position_y",
    "position_z",
]


with csv_path.open(
    "w",
    newline="",
    encoding="utf-8",
) as csv_file:

    writer = csv.DictWriter(
        csv_file,
        fieldnames=fieldnames,
    )

    writer.writeheader()

    for index in range(len(times)):

        writer.writerow(
            {
                "time": times[index],
                "command": command_active,

                "thrust_x": thrusts[index, 0],
                "thrust_y": thrusts[index, 1],
                "thrust_z": thrusts[index, 2],

                "drag_x": drags[index, 0],
                "drag_y": drags[index, 1],
                "drag_z": drags[index, 2],

                "gravity_x": gravities[index, 0],
                "gravity_y": gravities[index, 1],
                "gravity_z": gravities[index, 2],

                "buoyancy_x": buoyancies[index, 0],
                "buoyancy_y": buoyancies[index, 1],
                "buoyancy_z": buoyancies[index, 2],

                "net_force_x": net_forces[index, 0],
                "net_force_y": net_forces[index, 1],
                "net_force_z": net_forces[index, 2],

                "acceleration_x": accelerations[index, 0],
                "acceleration_y": accelerations[index, 1],
                "acceleration_z": accelerations[index, 2],

                "velocity_x": velocities[index, 0],
                "velocity_y": velocities[index, 1],
                "velocity_z": velocities[index, 2],

                "position_x": positions[index, 0],
                "position_y": positions[index, 1],
                "position_z": positions[index, 2],
            }
        )


# ---------------------------------------------------------------------
# Plot 1: static command-thrust response
# ---------------------------------------------------------------------

command_sweep = np.linspace(
    -1.5,
    1.5,
    301,
)

thrust_sweep = np.array([
    thruster_force_body(
        command=u,
        max_thrust=max_thrust,
        direction_body=direction_body,
    )[0]
    for u in command_sweep
])


fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.plot(
    command_sweep,
    thrust_sweep,
    linewidth=2,
)

ax.axvline(
    -1.0,
    linestyle="--",
    linewidth=1,
)

ax.axvline(
    1.0,
    linestyle="--",
    linewidth=1,
)

ax.axhline(
    -max_thrust,
    linestyle=":",
    linewidth=1,
)

ax.axhline(
    max_thrust,
    linestyle=":",
    linewidth=1,
)

ax.set_xlabel(
    "Normalized Thruster Command"
)

ax.set_ylabel(
    "Body-Frame X Thrust [N]"
)

ax.set_title(
    "Thruster Command-to-Force Response"
)

ax.grid(True)

fig.tight_layout()

fig.savefig(
    command_plot_path,
    dpi=160,
)

plt.close(fig)


# ---------------------------------------------------------------------
# Plot 2: velocity convergence
# ---------------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.plot(
    times,
    velocities[:, 0],
    linewidth=2,
    label="Numerical $v_x$",
)

ax.axhline(
    terminal_speed_analytic,
    linestyle="--",
    linewidth=1.5,
    label="Analytical terminal speed",
)

ax.set_xlabel(
    "Time [s]"
)

ax.set_ylabel(
    "Forward Velocity [m/s]"
)

ax.set_title(
    "Thruster-Driven Velocity Convergence"
)

ax.grid(True)

ax.legend()

fig.tight_layout()

fig.savefig(
    velocity_plot_path,
    dpi=160,
)

plt.close(fig)


# ---------------------------------------------------------------------
# Plot 3: horizontal force balance
# ---------------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.plot(
    times,
    thrusts[:, 0],
    linewidth=2,
    label="Thruster force",
)

ax.plot(
    times,
    drags[:, 0],
    linewidth=2,
    label="Drag force",
)

ax.plot(
    times,
    net_forces[:, 0],
    linewidth=2,
    label="Net force",
)

ax.axhline(
    0.0,
    linewidth=1,
)

ax.set_xlabel(
    "Time [s]"
)

ax.set_ylabel(
    "Force Along World X [N]"
)

ax.set_title(
    "Thruster-Drag Force Balance"
)

ax.grid(True)

ax.legend()

fig.tight_layout()

fig.savefig(
    force_plot_path,
    dpi=160,
)

plt.close(fig)


# ---------------------------------------------------------------------
# Artifact integrity checks
# ---------------------------------------------------------------------

expected_files = [
    csv_path,
    command_plot_path,
    velocity_plot_path,
    force_plot_path,
]


for path in expected_files:

    if not path.exists():
        raise FileNotFoundError(
            f"Expected artifact not created: {path}"
        )

    if path.stat().st_size == 0:
        raise RuntimeError(
            f"Generated artifact is empty: {path}"
        )


# ---------------------------------------------------------------------
# Reload CSV independently and validate its final state
# ---------------------------------------------------------------------

with csv_path.open(
    "r",
    newline="",
    encoding="utf-8",
) as csv_file:

    rows = list(
        csv.DictReader(csv_file)
    )


assert len(rows) == len(active_history)

final_row = rows[-1]

csv_final_velocity = np.array(
    [
        float(final_row["velocity_x"]),
        float(final_row["velocity_y"]),
        float(final_row["velocity_z"]),
    ]
)

csv_final_position = np.array(
    [
        float(final_row["position_x"]),
        float(final_row["position_y"]),
        float(final_row["position_z"]),
    ]
)

csv_final_net_force = np.array(
    [
        float(final_row["net_force_x"]),
        float(final_row["net_force_y"]),
        float(final_row["net_force_z"]),
    ]
)


np.testing.assert_allclose(
    csv_final_velocity,
    active_final["velocity"],
    rtol=0.0,
    atol=1e-12,
)

np.testing.assert_allclose(
    csv_final_position,
    active_final["position"],
    rtol=0.0,
    atol=1e-12,
)

np.testing.assert_allclose(
    csv_final_net_force,
    active_final["net_force"],
    rtol=0.0,
    atol=1e-12,
)

assert abs(
    csv_final_velocity[0]
    - terminal_speed_analytic
) < 1e-6


# ---------------------------------------------------------------------
# Report generated artifacts
# ---------------------------------------------------------------------

print("=" * 80)
print("NOTEBOOK 05 PERSISTENT ARTIFACTS")
print("=" * 80)

print(
    f"\nCSV samples: {len(rows)}"
)

print(
    "\nGenerated files:"
)

for path in expected_files:

    print(
        f"  [OK] "
        f"{path.relative_to(repo_root)} "
        f"({path.stat().st_size} bytes)"
    )


print(
    "\nReloaded CSV final velocity:",
    csv_final_velocity,
)

print(
    "Reloaded CSV final position:",
    csv_final_position,
)

print(
    "Reloaded CSV final net force:",
    csv_final_net_force,
)


print("\n" + "=" * 80)
print("STEP 6 PASSED")
print("Canonical Notebook 05 validation artifacts generated and verified.")
print("=" * 80)

NOTEBOOK 05 PERSISTENT ARTIFACTS

CSV samples: 2001

Generated files:
  [OK] results/notebooks/05_thruster_model/data/thruster_validation.csv (443063 bytes)
  [OK] results/notebooks/05_thruster_model/plots/command_thrust_response.png (60296 bytes)
  [OK] results/notebooks/05_thruster_model/plots/velocity_validation.png (61668 bytes)
  [OK] results/notebooks/05_thruster_model/plots/force_balance.png (61492 bytes)

Reloaded CSV final velocity: [1.99999999 0.         0.        ]
Reloaded CSV final position: [37.22353811  0.          0.        ]
Reloaded CSV final net force: [1.50146647e-07 0.00000000e+00 0.00000000e+00]

STEP 6 PASSED
Canonical Notebook 05 validation artifacts generated and verified.


## Step 7 - Promote the Validated Thruster Model Into Production Code

Steps 3 through 6 validated the reduced thruster model progressively from isolated command mapping to complete actuator-driven translational motion.

The notebook-local implementation has demonstrated that it:

- maps normalized commands into bounded thrust;
- saturates commands correctly at the physical actuator limits;
- supports bidirectional thrust;
- rejects invalid actuator parameters;
- enforces an explicit body-frame unit-direction contract;
- produces correct body-frame thrust vectors;
- rotates thrust correctly into the world frame;
- preserves thrust magnitude under rotation;
- integrates correctly with quadratic drag, gravity, and buoyancy;
- preserves neutral-buoyancy equilibrium at zero command;
- approaches the analytically predicted terminal speed under sustained thrust;
- introduces no unintended cross-axis motion;
- produces reproducible persistent validation artifacts.

The actuator model is therefore ready for production promotion.

---

### Production Module

The previously empty production target:

```text
src/hydrolab3d/dynamics/thrusters.py
```

will now contain two reusable functions:

```text
thruster_force_body()
thruster_force_world()
```

The first function implements the fundamental actuator mapping:

$$
\mathbf{F}_{T,B}
=
\operatorname{clip}(u,-1,1)
F_{max}
\hat{\mathbf{d}}_B
$$

The second combines that validated body-frame actuator model with the existing HydroLab-3D orientation infrastructure:

$$
\mathbf{F}_{T,W}
=
R_B^W
\mathbf{F}_{T,B}
$$

where:

$$
R_B^W
=
R_z(\psi)R_y(\theta)R_x(\phi)
$$

The world-frame function will reuse:

```text
rotation_body_to_world()
```

rather than duplicate rotation mathematics inside the thruster module.

---

### Production Parameter Contract

The production implementation preserves the validated notebook contract.

#### Command

The requested command must be:

- scalar;
- finite.

Commands may exceed the physical range because saturation is part of the actuator model:

$$
u_{sat}
=
\operatorname{clip}(u,-1,1)
$$

#### Maximum Thrust

Maximum thrust must satisfy:

$$
F_{max}>0
$$

and must be scalar and finite.

#### Thruster Direction

The body-frame thruster direction must:

- have shape `(3,)`;
- contain only finite values;
- not be the zero vector;
- already be a unit vector within numerical tolerance.

The production implementation will **not silently normalize** actuator directions.

This makes incorrectly configured thruster geometry visible rather than modifying it implicitly.

---

### Package Export

The HydroLab-3D dynamics package will expose:

```text
thruster_force_body
thruster_force_world
```

alongside the already validated dynamics capabilities:

```text
integrate_position
acceleration_from_force
integrate_velocity
quadratic_drag_force
gravity_force
buoyancy_force
```

through:

```text
src/hydrolab3d/dynamics/__init__.py
```

---

### Focused Regression Tests

The scientific notebook has already performed the detailed validation.

The automated regression suite should therefore protect the production contract without reproducing the entire notebook.

The thruster-specific test file:

```text
tests/test_thrusters.py
```

will protect the following behaviors:

1. zero command produces zero thrust;
2. proportional unsaturated command mapping is correct;
3. positive command saturation is correct;
4. negative command saturation is correct;
5. reverse thrust is correct;
6. body-frame direction is respected;
7. invalid actuator parameters are rejected;
8. identity orientation preserves thrust;
9. a known $90^\circ$ yaw produces the expected world-frame direction;
10. arbitrary rotation preserves thrust magnitude.

These tests protect the reusable actuator API while leaving the analytical terminal-speed and long-horizon integration experiments in the notebook where they belong.

---

### Source-Layout Test Environment

HydroLab-3D uses a source-layout repository:

```text
HydroLab-3D/
├── src/
│   └── hydrolab3d/
└── tests/
```

The notebook kernel already has:

```text
src/
```

on its Python path.

However, the `pytest` subprocess is a fresh Python process.

This step will therefore execute the focused thruster tests with:

```text
PYTHONPATH=<HydroLab-3D>/src
```

so that the regression test imports the production code that was just written.

---

### Expected Result

After this step:

```text
src/hydrolab3d/dynamics/thrusters.py
```

must contain the validated actuator implementation;

```text
src/hydrolab3d/dynamics/__init__.py
```

must expose the new thruster functions;

```text
tests/test_thrusters.py
```

must contain focused regression coverage;

and:

```text
pytest -q tests/test_thrusters.py
```

must pass from a fresh subprocess.

This step performs the production promotion and focused regression only.

The next and final implementation step will independently re-import the production implementation, rerun the canonical validation through production code, audit all Notebook 05 artifacts, and run the complete HydroLab-3D regression suite.

In [8]:
from pathlib import Path
import importlib
import os
import subprocess
import sys


# ---------------------------------------------------------------------
# Resolve repository paths
# ---------------------------------------------------------------------

repo_root = Path.cwd().resolve()

if repo_root.name == "notebooks":
    repo_root = repo_root.parent

src_path = repo_root / "src"

thruster_path = (
    src_path
    / "hydrolab3d"
    / "dynamics"
    / "thrusters.py"
)

dynamics_init_path = (
    src_path
    / "hydrolab3d"
    / "dynamics"
    / "__init__.py"
)

test_thruster_path = (
    repo_root
    / "tests"
    / "test_thrusters.py"
)


if not src_path.exists():
    raise FileNotFoundError(
        f"HydroLab-3D source directory not found: {src_path}"
    )


# ---------------------------------------------------------------------
# Verify inherited package state before modification
# ---------------------------------------------------------------------

if not dynamics_init_path.exists():
    raise FileNotFoundError(
        f"Dynamics package initializer not found: {dynamics_init_path}"
    )

existing_init_source = dynamics_init_path.read_text(
    encoding="utf-8"
)

required_inherited_names = [
    "buoyancy_force",
    "gravity_force",
    "quadratic_drag_force",
    "integrate_position",
    "acceleration_from_force",
    "integrate_velocity",
]

for name in required_inherited_names:
    assert name in existing_init_source, (
        f"Expected inherited dynamics export missing before promotion: {name}"
    )


# ---------------------------------------------------------------------
# Write validated production thruster module
# ---------------------------------------------------------------------

thruster_source = '''"""Reduced translational thruster models for HydroLab-3D."""

import numpy as np

from hydrolab3d.utils.transforms import rotation_body_to_world


def thruster_force_body(
    command,
    max_thrust,
    direction_body,
):
    """
    Convert a normalized thruster command into body-frame force.

    Parameters
    ----------
    command : float
        Requested normalized command. Values outside [-1, 1] are
        saturated.
    max_thrust : float
        Positive maximum thrust magnitude in newtons.
    direction_body : array-like, shape (3,)
        Body-frame unit vector defining positive thrust direction.

    Returns
    -------
    np.ndarray
        Body-frame thrust force [Fx, Fy, Fz] in newtons.
    """

    if (
        isinstance(command, (bool, np.bool_))
        or not np.isscalar(command)
    ):
        raise ValueError(
            "command must be a finite scalar."
        )

    command = float(command)

    if not np.isfinite(command):
        raise ValueError(
            "command must be finite."
        )

    if (
        isinstance(max_thrust, (bool, np.bool_))
        or not np.isscalar(max_thrust)
    ):
        raise ValueError(
            "max_thrust must be a finite positive scalar."
        )

    max_thrust = float(max_thrust)

    if (
        not np.isfinite(max_thrust)
        or max_thrust <= 0.0
    ):
        raise ValueError(
            "max_thrust must be finite and strictly positive."
        )

    direction_body = np.asarray(
        direction_body,
        dtype=float,
    )

    if direction_body.shape != (3,):
        raise ValueError(
            "direction_body must have shape (3,)."
        )

    if not np.all(
        np.isfinite(direction_body)
    ):
        raise ValueError(
            "direction_body must contain only finite values."
        )

    direction_norm = np.linalg.norm(
        direction_body
    )

    if direction_norm == 0.0:
        raise ValueError(
            "direction_body must not be the zero vector."
        )

    if not np.isclose(
        direction_norm,
        1.0,
        rtol=1e-9,
        atol=1e-12,
    ):
        raise ValueError(
            "direction_body must be a unit vector; "
            f"received norm {direction_norm:.12g}."
        )

    saturated_command = np.clip(
        command,
        -1.0,
        1.0,
    )

    thrust_magnitude = (
        saturated_command
        * max_thrust
    )

    return (
        thrust_magnitude
        * direction_body
    )


def thruster_force_world(
    command,
    max_thrust,
    direction_body,
    phi,
    theta,
    psi,
):
    """
    Convert a normalized thruster command into world-frame force.

    The thruster direction is defined in the vehicle body frame and
    transformed into the HydroLab-3D world frame using the validated
    body-to-world rotation convention.

    Parameters
    ----------
    command : float
        Requested normalized command.
    max_thrust : float
        Positive maximum thrust magnitude in newtons.
    direction_body : array-like, shape (3,)
        Body-frame unit vector defining positive thrust direction.
    phi : float
        Roll angle in radians.
    theta : float
        Pitch angle in radians.
    psi : float
        Yaw angle in radians.

    Returns
    -------
    np.ndarray
        World-frame thrust force [Fx, Fy, Fz] in newtons.
    """

    force_body = thruster_force_body(
        command=command,
        max_thrust=max_thrust,
        direction_body=direction_body,
    )

    rotation = rotation_body_to_world(
        phi=phi,
        theta=theta,
        psi=psi,
    )

    return rotation @ force_body
'''

thruster_path.write_text(
    thruster_source,
    encoding="utf-8",
)


# ---------------------------------------------------------------------
# Update dynamics package exports
# ---------------------------------------------------------------------

dynamics_init_source = '''"""Dynamics utilities for HydroLab-3D."""

from .buoyancy import buoyancy_force, gravity_force
from .drag import quadratic_drag_force
from .kinematics import integrate_position
from .rigid_body import acceleration_from_force, integrate_velocity
from .thrusters import thruster_force_body, thruster_force_world

__all__ = [
    "buoyancy_force",
    "gravity_force",
    "quadratic_drag_force",
    "integrate_position",
    "acceleration_from_force",
    "integrate_velocity",
    "thruster_force_body",
    "thruster_force_world",
]
'''

dynamics_init_path.write_text(
    dynamics_init_source,
    encoding="utf-8",
)


# ---------------------------------------------------------------------
# Write focused production regression tests
# ---------------------------------------------------------------------

test_source = '''import numpy as np
import pytest

from hydrolab3d.dynamics.thrusters import (
    thruster_force_body,
    thruster_force_world,
)


FORWARD = np.array([
    1.0,
    0.0,
    0.0,
])


def test_zero_command_gives_zero_thrust():
    force = thruster_force_body(
        command=0.0,
        max_thrust=100.0,
        direction_body=FORWARD,
    )

    np.testing.assert_allclose(
        force,
        np.zeros(3),
        rtol=0.0,
        atol=1e-12,
    )


def test_unsaturated_command_scales_linearly():
    force = thruster_force_body(
        command=0.25,
        max_thrust=100.0,
        direction_body=FORWARD,
    )

    np.testing.assert_allclose(
        force,
        np.array([25.0, 0.0, 0.0]),
        rtol=0.0,
        atol=1e-12,
    )


@pytest.mark.parametrize(
    "command, expected_x",
    [
        (2.0, 100.0),
        (-2.0, -100.0),
        (-0.5, -50.0),
    ],
)
def test_saturation_and_reverse_thrust(
    command,
    expected_x,
):
    force = thruster_force_body(
        command=command,
        max_thrust=100.0,
        direction_body=FORWARD,
    )

    np.testing.assert_allclose(
        force,
        np.array([expected_x, 0.0, 0.0]),
        rtol=0.0,
        atol=1e-12,
    )


def test_body_direction_is_respected():
    force = thruster_force_body(
        command=0.4,
        max_thrust=50.0,
        direction_body=np.array([
            0.0,
            0.0,
            1.0,
        ]),
    )

    np.testing.assert_allclose(
        force,
        np.array([0.0, 0.0, 20.0]),
        rtol=0.0,
        atol=1e-12,
    )


@pytest.mark.parametrize(
    "kwargs",
    [
        {
            "command": [0.5],
            "max_thrust": 100.0,
            "direction_body": FORWARD,
        },
        {
            "command": np.nan,
            "max_thrust": 100.0,
            "direction_body": FORWARD,
        },
        {
            "command": 0.5,
            "max_thrust": 0.0,
            "direction_body": FORWARD,
        },
        {
            "command": 0.5,
            "max_thrust": -1.0,
            "direction_body": FORWARD,
        },
        {
            "command": 0.5,
            "max_thrust": np.inf,
            "direction_body": FORWARD,
        },
        {
            "command": 0.5,
            "max_thrust": 100.0,
            "direction_body": [1.0, 0.0],
        },
        {
            "command": 0.5,
            "max_thrust": 100.0,
            "direction_body": [0.0, 0.0, 0.0],
        },
        {
            "command": 0.5,
            "max_thrust": 100.0,
            "direction_body": [1.0, np.nan, 0.0],
        },
        {
            "command": 0.5,
            "max_thrust": 100.0,
            "direction_body": [2.0, 0.0, 0.0],
        },
    ],
)
def test_invalid_parameters_are_rejected(kwargs):
    with pytest.raises(ValueError):
        thruster_force_body(**kwargs)


def test_identity_orientation_preserves_thrust():
    force = thruster_force_world(
        command=0.5,
        max_thrust=100.0,
        direction_body=FORWARD,
        phi=0.0,
        theta=0.0,
        psi=0.0,
    )

    np.testing.assert_allclose(
        force,
        np.array([50.0, 0.0, 0.0]),
        rtol=0.0,
        atol=1e-12,
    )


def test_positive_90_degree_yaw_rotates_forward_to_world_y():
    force = thruster_force_world(
        command=0.5,
        max_thrust=100.0,
        direction_body=FORWARD,
        phi=0.0,
        theta=0.0,
        psi=np.pi / 2.0,
    )

    np.testing.assert_allclose(
        force,
        np.array([0.0, 50.0, 0.0]),
        rtol=0.0,
        atol=1e-12,
    )


def test_arbitrary_rotation_preserves_thrust_magnitude():
    force = thruster_force_world(
        command=0.5,
        max_thrust=100.0,
        direction_body=FORWARD,
        phi=np.deg2rad(20.0),
        theta=np.deg2rad(-35.0),
        psi=np.deg2rad(70.0),
    )

    np.testing.assert_allclose(
        np.linalg.norm(force),
        50.0,
        rtol=1e-12,
        atol=1e-12,
    )
'''

test_thruster_path.write_text(
    test_source,
    encoding="utf-8",
)


# ---------------------------------------------------------------------
# Verify written production files
# ---------------------------------------------------------------------

written_files = [
    thruster_path,
    dynamics_init_path,
    test_thruster_path,
]

print("=" * 80)
print("NOTEBOOK 05 PRODUCTION PROMOTION")
print("=" * 80)

for path in written_files:

    assert path.exists()
    assert path.is_file()
    assert path.stat().st_size > 0

    print(
        "[OK]",
        path.relative_to(repo_root),
        f"({path.stat().st_size} bytes)",
    )


# ---------------------------------------------------------------------
# Reload package in notebook process
# ---------------------------------------------------------------------

src_path_str = str(src_path)

if src_path_str not in sys.path:
    sys.path.insert(0, src_path_str)

importlib.invalidate_caches()

import hydrolab3d.dynamics as dynamics

dynamics = importlib.reload(dynamics)


for name in [
    "integrate_position",
    "acceleration_from_force",
    "integrate_velocity",
    "quadratic_drag_force",
    "gravity_force",
    "buoyancy_force",
    "thruster_force_body",
    "thruster_force_world",
]:

    assert hasattr(
        dynamics,
        name,
    ), (
        f"Missing dynamics package export: {name}"
    )

    assert callable(
        getattr(dynamics, name)
    ), (
        f"Dynamics package export is not callable: {name}"
    )


print(
    "\n[OK] Dynamics package exports "
    "include both thruster functions."
)


# ---------------------------------------------------------------------
# Run focused thruster regression suite in fresh subprocess
# ---------------------------------------------------------------------

test_env = os.environ.copy()

existing_pythonpath = test_env.get(
    "PYTHONPATH",
    "",
)

if existing_pythonpath:
    test_env["PYTHONPATH"] = (
        str(src_path)
        + os.pathsep
        + existing_pythonpath
    )
else:
    test_env["PYTHONPATH"] = str(src_path)


print("\n" + "=" * 80)
print("FOCUSED THRUSTER REGRESSION SUITE")
print("=" * 80)

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
        "tests/test_thrusters.py",
    ],
    cwd=repo_root,
    env=test_env,
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

assert result.returncode == 0, (
    "Focused thruster regression suite failed."
)


print("=" * 80)
print("STEP 7 PASSED")
print("Validated thruster model promoted into production code.")
print("Focused production regression tests passed.")
print("=" * 80)

NOTEBOOK 05 PRODUCTION PROMOTION
[OK] src/hydrolab3d/dynamics/thrusters.py (3667 bytes)
[OK] src/hydrolab3d/dynamics/__init__.py (534 bytes)
[OK] tests/test_thrusters.py (3999 bytes)

[OK] Dynamics package exports include both thruster functions.

FOCUSED THRUSTER REGRESSION SUITE
..................                                                       [100%]
18 passed in 0.06s

STEP 7 PASSED
Validated thruster model promoted into production code.
Focused production regression tests passed.


## Step 8 - Production Re-Import, Canonical Revalidation, and Full Regression Audit

The validated thruster implementation has now been promoted into:

```text
src/hydrolab3d/dynamics/thrusters.py
```

and exposed through:

```text
src/hydrolab3d/dynamics/__init__.py
```

with focused regression protection in:

```text
tests/test_thrusters.py
```

The focused actuator suite has passed:

```text
18 passed
```

The final implementation step must now verify that Notebook 05 no longer depends on its notebook-local prototype.

This step performs three independent checks:

1. re-import the promoted production thruster implementation;
2. rerun the canonical actuator-driven translational experiment using production code;
3. run the complete HydroLab-3D regression suite.

No further production modifications are expected.

---

### Production Re-Import

The notebook will explicitly import:

```text
thruster_force_body
thruster_force_world
```

from the production package.

The canonical validation must then use these imported functions rather than the notebook-local definitions created during prototyping.

This establishes the final development lineage:

```text
notebook prototype
      ↓
isolated validation
      ↓
integrated physical validation
      ↓
production promotion
      ↓
production re-import
      ↓
canonical revalidation
```

---

### Canonical Production Experiment

The final production validation retains the same canonical configuration used in Step 5:

$$
m=20\text{ kg}
$$

$$
\rho=1000\text{ kg/m}^3
$$

$$
V=\frac{m}{\rho}=0.020\text{ m}^3
$$

$$
C_D=5
$$

$$
F_{max}=40\text{ N}
$$

$$
u=0.5
$$

with a forward-facing body-frame thruster:

$$
\hat{\mathbf{d}}_B
=
\begin{bmatrix}
1\\
0\\
0
\end{bmatrix}
$$

and zero vehicle orientation.

The resulting steady thrust is:

$$
F_T=20\text{ N}
$$

The analytical terminal speed remains:

$$
v_{x,\infty}
=
\sqrt{\frac{F_T}{C_D}}
=
2\text{ m/s}
$$

The production implementation should reproduce the already validated notebook result:

$$
v_{x,\text{final}}
\approx
1.999999992493\text{ m/s}
$$

within numerical tolerance.

---

### Production Equivalence Requirements

The production rerun must confirm:

- zero command still produces zero thrust;
- command saturation still behaves correctly;
- the promoted world-frame actuator produces the expected forward force;
- gravity and buoyancy still cancel for the neutral vehicle;
- the vehicle approaches the analytical terminal speed;
- final thrust and drag approximately balance;
- no unintended $y$ or $z$ motion appears;
- all simulated values remain finite.

The final state should remain consistent with the previously validated notebook-local experiment.

Any meaningful disagreement would indicate that promotion changed behavior and must be investigated before Notebook 05 can conclude.

---

### Artifact Audit

Notebook 05 has already generated:

```text
results/notebooks/05_thruster_model/
├── data/
│   └── thruster_validation.csv
└── plots/
    ├── command_thrust_response.png
    ├── velocity_validation.png
    └── force_balance.png
```

The final audit will verify that all four artifacts still:

- exist;
- are non-empty.

The canonical CSV should contain:

```text
2001 samples
```

from the validated 20-second experiment at:

$$
\Delta t=0.01\text{ s}
$$

including both the initial state and final state.

No additional artifact is required unless this final audit uncovers a problem.

---

### Complete Repository Regression

Notebook 04 ended with:

```text
51 passed
```

Notebook 05 has now added a focused thruster regression suite containing 18 passing test cases.

The complete suite will be executed from the repository root using the production source tree.

The exact final test count will be taken from the actual `pytest` result rather than assumed in advance.

This is important because the acceptance criterion is not a predicted test count.

The acceptance criterion is:

```text
all collected HydroLab-3D tests pass
```

with no regression in the previously validated kinematics, orientation, dynamics, drag, gravity, or buoyancy components.

---

### Final Acceptance Criteria

Notebook 05 passes its final implementation step only if:

1. the promoted thruster functions import from production code;
2. their source location resolves to the production `thrusters.py` module;
3. the production implementation reproduces the canonical terminal-speed experiment;
4. numerical final speed agrees with:

$$
v_{x,\infty}=2\text{ m/s}
$$

5. final thrust and drag are in equilibrium to the established numerical tolerance;
6. no cross-axis motion appears;
7. all persistent Notebook 05 artifacts exist and are non-empty;
8. the canonical CSV contains the expected simulation history;
9. the complete HydroLab-3D `pytest` suite passes.

If this step passes, no further implementation code is required.

Notebook 05 will then be complete and ready for its final Conclusion Markdown cell and engineering handoff to:

```text
06_ocean_currents.ipynb
```

In [9]:
import csv
import importlib
import inspect
import os
from pathlib import Path
import subprocess
import sys

import numpy as np


# ---------------------------------------------------------------------
# Resolve repository and production package
# ---------------------------------------------------------------------

repo_root = Path.cwd().resolve()

if repo_root.name == "notebooks":
    repo_root = repo_root.parent

src_path = repo_root / "src"

src_path_str = str(src_path)

if src_path_str not in sys.path:
    sys.path.insert(0, src_path_str)

importlib.invalidate_caches()


# ---------------------------------------------------------------------
# Explicitly import promoted production implementation
# ---------------------------------------------------------------------

import hydrolab3d.dynamics.thrusters as production_thrusters

production_thrusters = importlib.reload(
    production_thrusters
)

from hydrolab3d.dynamics import (
    acceleration_from_force as production_acceleration_from_force,
    buoyancy_force as production_buoyancy_force,
    gravity_force as production_gravity_force,
    integrate_position as production_integrate_position,
    integrate_velocity as production_integrate_velocity,
    quadratic_drag_force as production_quadratic_drag_force,
    thruster_force_body as production_thruster_force_body,
    thruster_force_world as production_thruster_force_world,
)


# ---------------------------------------------------------------------
# Verify actual implementation source
# ---------------------------------------------------------------------

expected_thruster_path = (
    src_path
    / "hydrolab3d"
    / "dynamics"
    / "thrusters.py"
).resolve()

body_source_path = Path(
    inspect.getsourcefile(
        production_thruster_force_body
    )
).resolve()

world_source_path = Path(
    inspect.getsourcefile(
        production_thruster_force_world
    )
).resolve()


assert body_source_path == expected_thruster_path
assert world_source_path == expected_thruster_path


print("=" * 80)
print("PRODUCTION THRUSTER RE-IMPORT")
print("=" * 80)

print(
    f"thruster_force_body  -> "
    f"{body_source_path.relative_to(repo_root)}"
)

print(
    f"thruster_force_world -> "
    f"{world_source_path.relative_to(repo_root)}"
)

print("[PASS] Notebook is using promoted production actuator code.")


# =====================================================================
# BASIC PRODUCTION ACTUATOR SANITY CHECKS
# =====================================================================

forward_body = np.array(
    [1.0, 0.0, 0.0],
    dtype=float,
)


zero_thrust = production_thruster_force_body(
    command=0.0,
    max_thrust=40.0,
    direction_body=forward_body,
)

positive_saturation = production_thruster_force_body(
    command=2.0,
    max_thrust=40.0,
    direction_body=forward_body,
)

negative_saturation = production_thruster_force_body(
    command=-2.0,
    max_thrust=40.0,
    direction_body=forward_body,
)


np.testing.assert_allclose(
    zero_thrust,
    np.zeros(3),
    rtol=0.0,
    atol=1e-12,
)

np.testing.assert_allclose(
    positive_saturation,
    np.array([40.0, 0.0, 0.0]),
    rtol=0.0,
    atol=1e-12,
)

np.testing.assert_allclose(
    negative_saturation,
    np.array([-40.0, 0.0, 0.0]),
    rtol=0.0,
    atol=1e-12,
)


print("\nProduction actuator sanity checks:")
print(f"  zero command        : {zero_thrust} [PASS]")
print(f"  positive saturation : {positive_saturation} [PASS]")
print(f"  negative saturation : {negative_saturation} [PASS]")


# =====================================================================
# CANONICAL PRODUCTION DYNAMICS EXPERIMENT
# =====================================================================

mass = 20.0
fluid_density = 1000.0
displaced_volume = mass / fluid_density
gravity = 9.81

drag_coefficient = 5.0

max_thrust = 40.0
command = 0.5

phi = 0.0
theta = 0.0
psi = 0.0

dt = 0.01
duration = 20.0
steps = int(duration / dt)


force_gravity = production_gravity_force(
    mass,
    gravity,
)

force_buoyancy = production_buoyancy_force(
    fluid_density,
    displaced_volume,
    gravity,
)


np.testing.assert_allclose(
    force_gravity + force_buoyancy,
    np.zeros(3),
    rtol=0.0,
    atol=1e-12,
)


position = np.zeros(3, dtype=float)
velocity = np.zeros(3, dtype=float)

production_history = []


for step in range(steps + 1):

    time = step * dt

    force_thrust = production_thruster_force_world(
        command=command,
        max_thrust=max_thrust,
        direction_body=forward_body,
        phi=phi,
        theta=theta,
        psi=psi,
    )

    force_drag = production_quadratic_drag_force(
        velocity,
        drag_coefficient,
    )

    force_net = (
        force_thrust
        + force_drag
        + force_gravity
        + force_buoyancy
    )

    acceleration = production_acceleration_from_force(
        force_net,
        mass,
    )

    production_history.append(
        {
            "time": time,
            "position": position.copy(),
            "velocity": velocity.copy(),
            "thrust": force_thrust.copy(),
            "drag": force_drag.copy(),
            "gravity": force_gravity.copy(),
            "buoyancy": force_buoyancy.copy(),
            "net_force": force_net.copy(),
            "acceleration": acceleration.copy(),
        }
    )

    if step == steps:
        break

    position = production_integrate_position(
        position,
        velocity,
        dt,
    )

    velocity = production_integrate_velocity(
        velocity,
        acceleration,
        dt,
    )


production_final = production_history[-1]


# ---------------------------------------------------------------------
# Analytical validation
# ---------------------------------------------------------------------

expected_thrust = command * max_thrust

terminal_speed_analytic = np.sqrt(
    expected_thrust
    / drag_coefficient
)

terminal_speed_production = (
    production_final["velocity"][0]
)

terminal_speed_error = abs(
    terminal_speed_production
    - terminal_speed_analytic
)


np.testing.assert_allclose(
    production_final["thrust"],
    np.array([
        expected_thrust,
        0.0,
        0.0,
    ]),
    rtol=0.0,
    atol=1e-12,
)

assert terminal_speed_error < 1e-6

assert abs(
    production_final["net_force"][0]
) < 1e-5

np.testing.assert_allclose(
    production_final["velocity"][1:],
    np.zeros(2),
    rtol=0.0,
    atol=1e-12,
)

np.testing.assert_allclose(
    production_final["position"][1:],
    np.zeros(2),
    rtol=0.0,
    atol=1e-12,
)


# ---------------------------------------------------------------------
# Compare production rerun with validated notebook-local Step 5 result
# ---------------------------------------------------------------------

np.testing.assert_allclose(
    production_final["velocity"],
    active_final["velocity"],
    rtol=0.0,
    atol=1e-12,
)

np.testing.assert_allclose(
    production_final["position"],
    active_final["position"],
    rtol=0.0,
    atol=1e-12,
)

np.testing.assert_allclose(
    production_final["net_force"],
    active_final["net_force"],
    rtol=0.0,
    atol=1e-12,
)


# ---------------------------------------------------------------------
# Finite-value audit
# ---------------------------------------------------------------------

for record in production_history:

    for name in (
        "position",
        "velocity",
        "thrust",
        "drag",
        "gravity",
        "buoyancy",
        "net_force",
        "acceleration",
    ):

        if not np.all(
            np.isfinite(record[name])
        ):
            raise AssertionError(
                f"Non-finite production {name} "
                f"at t={record['time']}"
            )


print("\n" + "=" * 80)
print("CANONICAL PRODUCTION REVALIDATION")
print("=" * 80)

print(
    f"Analytical terminal speed : "
    f"{terminal_speed_analytic:.12f} m/s"
)

print(
    f"Production final speed    : "
    f"{terminal_speed_production:.12f} m/s"
)

print(
    f"Absolute speed error      : "
    f"{terminal_speed_error:.12e} m/s"
)

print(
    f"Production final position : "
    f"{production_final['position']}"
)

print(
    f"Production final thrust   : "
    f"{production_final['thrust']}"
)

print(
    f"Production final drag     : "
    f"{production_final['drag']}"
)

print(
    f"Production final net force: "
    f"{production_final['net_force']}"
)

print(
    f"Production final accel.   : "
    f"{production_final['acceleration']}"
)

print(
    "[PASS] Production implementation exactly reproduces "
    "the validated notebook-local canonical result."
)


# =====================================================================
# NOTEBOOK 05 ARTIFACT AUDIT
# =====================================================================

output_root = (
    repo_root
    / "results"
    / "notebooks"
    / "05_thruster_model"
)

csv_path = (
    output_root
    / "data"
    / "thruster_validation.csv"
)

command_plot_path = (
    output_root
    / "plots"
    / "command_thrust_response.png"
)

velocity_plot_path = (
    output_root
    / "plots"
    / "velocity_validation.png"
)

force_plot_path = (
    output_root
    / "plots"
    / "force_balance.png"
)


expected_artifacts = [
    csv_path,
    command_plot_path,
    velocity_plot_path,
    force_plot_path,
]


print("\n" + "=" * 80)
print("NOTEBOOK 05 ARTIFACT AUDIT")
print("=" * 80)


for path in expected_artifacts:

    if not path.exists():
        raise FileNotFoundError(
            f"Missing Notebook 05 artifact: {path}"
        )

    if not path.is_file():
        raise RuntimeError(
            f"Expected file but found another path type: {path}"
        )

    if path.stat().st_size == 0:
        raise RuntimeError(
            f"Notebook 05 artifact is empty: {path}"
        )

    print(
        f"[OK] "
        f"{path.relative_to(repo_root)} "
        f"({path.stat().st_size} bytes)"
    )


# ---------------------------------------------------------------------
# Reload canonical CSV
# ---------------------------------------------------------------------

with csv_path.open(
    "r",
    newline="",
    encoding="utf-8",
) as csv_file:

    csv_rows = list(
        csv.DictReader(csv_file)
    )


assert len(csv_rows) == len(
    production_history
)

assert len(csv_rows) == 2001


csv_final_row = csv_rows[-1]

csv_final_velocity = np.array(
    [
        float(csv_final_row["velocity_x"]),
        float(csv_final_row["velocity_y"]),
        float(csv_final_row["velocity_z"]),
    ]
)

csv_final_position = np.array(
    [
        float(csv_final_row["position_x"]),
        float(csv_final_row["position_y"]),
        float(csv_final_row["position_z"]),
    ]
)


np.testing.assert_allclose(
    csv_final_velocity,
    production_final["velocity"],
    rtol=0.0,
    atol=1e-12,
)

np.testing.assert_allclose(
    csv_final_position,
    production_final["position"],
    rtol=0.0,
    atol=1e-12,
)


print(
    f"\nCSV samples               : "
    f"{len(csv_rows)}"
)

print(
    f"CSV final velocity        : "
    f"{csv_final_velocity}"
)

print(
    f"CSV final position        : "
    f"{csv_final_position}"
)

print(
    "[PASS] Persistent canonical data remains consistent "
    "with the production rerun."
)


# =====================================================================
# COMPLETE HYDROLAB-3D REGRESSION SUITE
# =====================================================================

test_env = os.environ.copy()

existing_pythonpath = test_env.get(
    "PYTHONPATH",
    "",
)

if existing_pythonpath:

    test_env["PYTHONPATH"] = (
        str(src_path)
        + os.pathsep
        + existing_pythonpath
    )

else:

    test_env["PYTHONPATH"] = str(
        src_path
    )


print("\n" + "=" * 80)
print("COMPLETE HYDROLAB-3D REGRESSION SUITE")
print("=" * 80)


full_test_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
    ],
    cwd=repo_root,
    env=test_env,
    text=True,
    capture_output=True,
)


print(full_test_result.stdout)

if full_test_result.stderr:
    print(full_test_result.stderr)


assert full_test_result.returncode == 0, (
    "Complete HydroLab-3D regression suite failed."
)


# ---------------------------------------------------------------------
# Final Step 8 status
# ---------------------------------------------------------------------

print("=" * 80)
print("STEP 8 PASSED")
print("Production thruster implementation revalidated successfully.")
print("Notebook 05 persistent artifacts verified.")
print("Complete HydroLab-3D regression suite passed.")
print("=" * 80)

PRODUCTION THRUSTER RE-IMPORT
thruster_force_body  -> src/hydrolab3d/dynamics/thrusters.py
thruster_force_world -> src/hydrolab3d/dynamics/thrusters.py
[PASS] Notebook is using promoted production actuator code.

Production actuator sanity checks:
  zero command        : [0. 0. 0.] [PASS]
  positive saturation : [40.  0.  0.] [PASS]
  negative saturation : [-40.  -0.  -0.] [PASS]

CANONICAL PRODUCTION REVALIDATION
Analytical terminal speed : 2.000000000000 m/s
Production final speed    : 1.999999992493 m/s
Absolute speed error      : 7.507332311363e-09 m/s
Production final position : [37.22353811  0.          0.        ]
Production final thrust   : [20.  0.  0.]
Production final drag     : [-19.99999985  -0.          -0.        ]
Production final net force: [1.50146647e-07 0.00000000e+00 0.00000000e+00]
Production final accel.   : [7.50733236e-09 0.00000000e+00 0.00000000e+00]
[PASS] Production implementation exactly reproduces the validated notebook-local canonical result.

NOTEBOOK 0

# Conclusion

## What Was Implemented

Notebook 05 introduced a reduced-order thruster model into the validated HydroLab-3D translational dynamics stack.

The actuator layer now converts a normalized scalar command into a bounded physical thrust force:

$$
u_{sat}
=
\operatorname{clip}(u,-1,1)
$$

$$
F_T
=
u_{sat}F_{max}
$$

For a body-frame unit thruster direction:

$$
\hat{\mathbf{d}}_B
$$

the body-frame thrust vector is:

$$
\mathbf{F}_{T,B}
=
F_T\hat{\mathbf{d}}_B
$$

and the corresponding world-frame force is:

$$
\mathbf{F}_{T,W}
=
R_B^W
\mathbf{F}_{T,B}
$$

using the already validated HydroLab-3D body-to-world rotation:

$$
R_B^W
=
R_z(\psi)R_y(\theta)R_x(\phi)
$$

The resulting reduced translational force balance is now:

$$
m\dot{\mathbf{v}}
=
\mathbf{F}_T
+
\mathbf{F}_D
+
\mathbf{F}_G
+
\mathbf{F}_B
$$

with the existing explicit-Euler integration convention retained.

The production actuator implementation provides:

```text
thruster_force_body()
thruster_force_world()
```

in:

```text
src/hydrolab3d/dynamics/thrusters.py
```

---

## What Was Validated

Notebook 05 validated the actuator model progressively from isolated command mapping through full translational vehicle dynamics.

### Command-to-Thrust Mapping

The notebook confirmed that:

- zero command produces zero thrust;
- positive commands produce thrust in the configured direction;
- negative commands reverse the thrust direction;
- thrust scales linearly within the unsaturated command region;
- commands greater than $1$ saturate at maximum forward thrust;
- commands less than $-1$ saturate at maximum reverse thrust.

For the initial validation actuator:

$$
F_{max}=100\text{ N}
$$

representative results included:

```text
u =  0.00  →    0 N
u =  0.25  →   25 N
u =  0.50  →   50 N
u =  1.00  →  100 N
u =  2.00  →  100 N

u = -0.50  →  -50 N
u = -1.00  → -100 N
u = -2.00  → -100 N
```

### Parameter Validation

The actuator rejected representative invalid inputs including:

- non-scalar commands;
- non-finite commands;
- zero maximum thrust;
- negative maximum thrust;
- non-finite maximum thrust;
- incorrectly shaped direction vectors;
- zero-magnitude direction vectors;
- non-finite direction components;
- non-unit direction vectors.

The thruster API deliberately requires a valid body-frame unit vector and does not silently normalize incorrectly configured actuator directions.

### Reference-Frame Behavior

A forward-facing body-frame thruster:

$$
\hat{\mathbf{d}}_B
=
\begin{bmatrix}
1\\
0\\
0
\end{bmatrix}
$$

was validated under several orientations.

At zero orientation:

$$
\mathbf{F}_{T,W}
=
\begin{bmatrix}
50\\
0\\
0
\end{bmatrix}
\text{ N}
$$

A positive $90^\circ$ yaw produced approximately:

$$
\mathbf{F}_{T,W}
=
\begin{bmatrix}
0\\
50\\
0
\end{bmatrix}
\text{ N}
$$

and a positive $90^\circ$ pitch produced approximately:

$$
\mathbf{F}_{T,W}
=
\begin{bmatrix}
0\\
0\\
-50
\end{bmatrix}
\text{ N}
$$

The tiny residual components on the order of:

$$
10^{-15}
$$

were ordinary floating-point roundoff.

For the arbitrary orientation test:

```text
roll  =  20 deg
pitch = -35 deg
yaw   =  70 deg
```

the body-frame force:

```text
[50, 0, 0] N
```

became approximately:

```text
[14.00832498, 38.48755657, 28.67882182] N
```

while preserving the force magnitude exactly to numerical tolerance:

```text
body magnitude  = 50.000000000000 N
world magnitude = 50.000000000000 N
```

Applying the transpose rotation recovered the original body-frame force to floating-point precision.

### Complete Translational Integration

The thruster was then coupled with the validated:

- quadratic drag;
- gravity;
- buoyancy;
- force-to-acceleration conversion;
- velocity integration;
- position propagation.

A neutrally buoyant vehicle with zero thruster command remained exactly stationary:

```text
Final position  = [0, 0, 0] m
Final velocity  = [0, 0, 0] m/s
Final net force = [0, 0, 0] N
```

No spontaneous motion was introduced by the actuator layer.

---

## Key Results

The canonical thrust-driven validation used:

$$
m=20\text{ kg}
$$

$$
\rho=1000\text{ kg/m}^3
$$

$$
V=0.020\text{ m}^3
$$

$$
C_D=5
$$

$$
F_{max}=40\text{ N}
$$

$$
u=0.5
$$

giving constant thrust:

$$
F_T=20\text{ N}
$$

The vehicle was neutrally buoyant:

$$
\mathbf{F}_G
=
\begin{bmatrix}
0\\
0\\
-196.2
\end{bmatrix}
\text{ N}
$$

$$
\mathbf{F}_B
=
\begin{bmatrix}
0\\
0\\
196.2
\end{bmatrix}
\text{ N}
$$

so:

$$
\mathbf{F}_G+\mathbf{F}_B
=
\mathbf{0}
$$

For one-dimensional forward motion with quadratic drag:

$$
F_D=-C_Dv_x^2
$$

the analytical terminal-speed condition is:

$$
F_T+F_D=0
$$

giving:

$$
v_{x,\infty}
=
\sqrt{\frac{F_T}{C_D}}
$$

and therefore:

$$
v_{x,\infty}
=
\sqrt{\frac{20}{5}}
=
2.000000000000\text{ m/s}
$$

The production simulation produced:

```text
Analytical terminal speed : 2.000000000000 m/s
Numerical final speed     : 1.999999992493 m/s
Absolute speed error      : 7.507332311363e-09 m/s
```

The final vehicle position after the 20-second validation was:

```text
[37.22353811, 0, 0] m
```

The final horizontal force state was:

```text
Thruster force =  20.00000000 N
Drag force     = -19.99999985 N
Net force      =   1.50146647e-07 N
Acceleration   =   7.50733236e-09 m/s²
```

Thus:

$$
F_T+F_D
\rightarrow
0
$$

and:

$$
a_x
\rightarrow
0
$$

as the vehicle approached terminal speed.

No unintended $y$ or $z$ motion appeared.

---

## Production-Code Promotion

After notebook-local validation, the reusable actuator implementation was promoted into:

```text
src/hydrolab3d/dynamics/thrusters.py
```

with:

```text
thruster_force_body()
thruster_force_world()
```

The dynamics package exports were updated in:

```text
src/hydrolab3d/dynamics/__init__.py
```

and focused regression protection was added to:

```text
tests/test_thrusters.py
```

The focused thruster suite completed with:

```text
18 passed
```

The notebook then explicitly re-imported the production actuator implementation and verified that both functions resolved to:

```text
src/hydrolab3d/dynamics/thrusters.py
```

The canonical experiment was rerun through production code and reproduced the notebook-local result exactly within the established numerical tolerances.

The complete HydroLab-3D regression suite then completed with:

```text
69 passed
```

confirming that the thruster integration introduced no regression into the previously validated simulator components.

---

## Generated Outputs

Notebook 05 generated and verified the following persistent artifacts:

```text
results/notebooks/05_thruster_model/
├── data/
│   └── thruster_validation.csv
└── plots/
    ├── command_thrust_response.png
    ├── velocity_validation.png
    └── force_balance.png
```

### Validation Dataset

```text
results/notebooks/05_thruster_model/data/thruster_validation.csv
```

contains:

```text
2001 samples
```

covering the complete 20-second canonical simulation with:

$$
\Delta t=0.01\text{ s}
$$

The dataset records:

- time;
- command;
- thruster force;
- drag force;
- gravity;
- buoyancy;
- net force;
- acceleration;
- velocity;
- position.

Its final state was independently reloaded and verified against the production rerun.

### Command-Thrust Response

```text
results/notebooks/05_thruster_model/plots/command_thrust_response.png
```

demonstrates the linear command region and actuator saturation outside:

$$
u\in[-1,1]
$$

### Velocity Validation

```text
results/notebooks/05_thruster_model/plots/velocity_validation.png
```

shows the simulated forward velocity converging toward the analytical terminal-speed prediction:

$$
v_{x,\infty}=2\text{ m/s}
$$

### Force Balance

```text
results/notebooks/05_thruster_model/plots/force_balance.png
```

shows the evolving relationship between:

- thruster force;
- opposing quadratic drag;
- net horizontal force.

It demonstrates:

$$
F_T+F_D\rightarrow0
$$

as terminal velocity is approached.

---

## Output Location

All persistent Notebook 05 validation artifacts are stored under:

```text
results/notebooks/05_thruster_model/
```

The final verified artifact set is:

```text
results/notebooks/05_thruster_model/data/thruster_validation.csv

results/notebooks/05_thruster_model/plots/command_thrust_response.png

results/notebooks/05_thruster_model/plots/velocity_validation.png

results/notebooks/05_thruster_model/plots/force_balance.png
```

---

## Important Observations

The reduced actuator layer was sufficient to establish a physically meaningful command-to-motion chain without introducing unnecessary propulsion-system complexity.

Several design decisions are important for future HydroLab-3D development.

### Saturation Belongs Inside the Actuator Model

The requested command may exceed the physical command range, but generated thrust remains bounded:

$$
F_T
\in
[-F_{max},F_{max}]
$$

This allows future controllers to issue commands without bypassing actuator limits.

### Thruster Geometry Is Explicit

Thruster direction is defined in the body frame rather than directly in world coordinates.

Therefore, a fixed physical thruster automatically changes its world-frame force direction when the vehicle rotates.

This preserves the causal relationship:

```text
vehicle orientation
        ↓
thruster orientation in world frame
        ↓
world-frame actuator force
```

### Rotation Does Not Change Thrust Magnitude

The validated orientation transformation changed only force direction:

$$
\|\mathbf{F}_{T,W}\|
=
\|\mathbf{F}_{T,B}\|
$$

No artificial actuator energy was introduced by reference-frame transformation.

### Neutral Buoyancy Provides a Clean Horizontal Validation Case

Explicitly retaining gravity and buoyancy while choosing:

$$
\rho V=m
$$

allowed the complete force balance to remain active without contaminating the horizontal thruster-drag experiment.

### The Terminal-Speed Experiment Validates More Than the Thruster Alone

The close agreement between:

```text
Analytical : 2.000000000000 m/s
Numerical  : 1.999999992493 m/s
```

simultaneously validates the interaction among:

```text
thruster
+
quadratic drag
+
force summation
+
mass
+
acceleration
+
velocity integration
```

rather than testing the actuator in isolation.

---

## Current Limitations

Notebook 05 intentionally implements a reduced static actuator model.

It does not yet model:

- actuator first-order lag;
- thrust slew-rate limits;
- motor RPM dynamics;
- propeller dynamics;
- asymmetric forward/reverse thrust capability;
- thrust curves identified from hardware;
- voltage or battery effects;
- cavitation;
- water-density dependence of propeller performance;
- thruster efficiency variation;
- multiple-thruster geometry;
- actuator allocation;
- actuator-generated torque;
- rotational dynamics;
- full 6-DOF rigid-body dynamics;
- added mass;
- Coriolis or centripetal effects;
- ocean-current-relative inflow;
- actuator faults or degradation.

These omissions are deliberate.

Notebook 05 establishes the smallest trustworthy actuator layer required by the current reduced translational simulator.

Higher actuator fidelity can be added later if a downstream control or sim-to-real problem demonstrates that it materially affects the research question.

---

## How This Result Will Be Used

HydroLab-3D now has a validated translational dynamics chain:

```text
normalized command
        ↓
thruster saturation
        ↓
body-frame thrust
        ↓
body-to-world transformation
        ↓
world-frame thrust
        ↓
drag + gravity + buoyancy
        ↓
net force
        ↓
acceleration
        ↓
velocity
        ↓
position
```

The governing reduced-order dynamics are:

$$
m\dot{\mathbf{v}}
=
\mathbf{F}_T
+
\mathbf{F}_D
+
\mathbf{F}_G
+
\mathbf{F}_B
$$

with actuator force:

$$
\mathbf{F}_{T,W}
=
R_B^W
\left[
\operatorname{clip}(u,-1,1)
F_{max}
\hat{\mathbf{d}}_B
\right]
$$

Future simulator components can therefore command physical actuators rather than directly injecting arbitrary vehicle forces or manipulating state.

The validated actuator layer can later support:

- manual vehicle commands;
- closed-loop controllers;
- waypoint navigation;
- multi-thruster systems;
- full 6-DOF vehicle dynamics;
- learning-based policies;
- actuator-failure experiments;
- ROS 2 actuator interfaces.

---

## Next Notebook

The next development notebook is:

```text
06_ocean_currents.ipynb
```

Notebook 06 can now begin from the validated translational stack:

$$
m\dot{\mathbf{v}}
=
\mathbf{F}_T
+
\mathbf{F}_D
+
\mathbf{F}_G
+
\mathbf{F}_B
$$

and introduce environmental flow and current effects without reopening the fundamental validation of:

- 3D kinematics;
- orientation transforms;
- force-to-acceleration conversion;
- velocity integration;
- quadratic drag;
- gravity;
- buoyancy;
- command-to-thrust conversion;
- command saturation;
- thruster direction;
- body-to-world thrust transformation.

Notebook 05 therefore completes the actuator layer required before environmental disturbances are introduced.

The engineering handoff is now:

```text
Notebook 00
Validated 3D kinematics
        ↓
Notebook 01
Validated orientation and reference frames
        ↓
Notebook 02
Validated force-driven translational dynamics
        ↓
Notebook 03
Validated quadratic hydrodynamic drag
        ↓
Notebook 04
Validated gravity and buoyancy
        ↓
Notebook 05
Validated command-driven thruster actuation
        ↓
Notebook 06
Ocean currents and environmental flow
```

HydroLab-3D now has a validated actuator-driven reduced translational vehicle model ready for environmental-flow development.

---